# Kennicutt–Schmidt evolution tracks — cis25 quenched sample

Where do SIMBA's quenched galaxies travel on the **Kennicutt–Schmidt plane**
(log Σ$_{\rm H_2}$ vs log Σ$_{\rm SFR}$) while they quench, and where do they sit compared with the
observed ALMA-C11 quiescent galaxies (z ≈ 0.34–0.43, `obs_data/almac11/ks_table.csv`)?

Sample: the 266 quenched galaxies of `powderday_flux_quenched_m25.ipynb` (10 anchors, z = 0.3–2,
`tables/powderday_quenched_selection.fits`) plus their mass-matched star-forming partners as a
reference cloud at the anchor. Every galaxy is followed through its **critical epochs**
(sSFR peak → SFT quench start → QT quench end → post-quench → H$_2$ trough → anchor) via the
per-anchor histories + progen links, and Σ$_{\rm H_2}$ / Σ$_{\rm SFR}$ are measured from the
**CAESAR member particles (glist / slist)** stored in the reduced particle files
(`build_reduced_particles_job.py`) at each epoch.

Conventions (details in the closing cell):
* **fiducial Σ**: inside the galaxy's own face-on H$_2$ half-mass radius, Σ = 0.5 M / (π R$_{50}^2$)
  — the observed convention (0.5 M$_{\rm H_2}$ / π R$_{\rm CO}^2$, same R for the SFR);
* **fiducial SFR**: archaeological, stars formed in the last 100 Myr (slist); the instantaneous gas
  SFR (glist) and a 25 Myr window are stored too; SFR = 0 → one-particle upper limit;
* **drawn track** = quench start (SFT) → quench end (QT) → **track end**: the anchor, or — when the anchor
  holds no measurable H$_2$ (catalogue M$_{\rm H_2}$/M$_\star$ ≤ 10$^{-4}$) — the last snapshot with M$_{\rm H_2}$/M$_\star$ > 10$^{-4}$
  (stage `end`, `FH2_MIN_END`); no other stage is required, so every galaxy with a quench event contributes;
* fixed apertures r < 1 / 3.16 / 10 kpc (the m25 ladder, no 0.5 factor) are measured and summarised but
  **not plotted**: a fixed aperture is a different Σ definition from the observed 0.5 M/πR$^2$ one;
* simulated H$_2$ ×1.36 (He) for the comparison; Planck15 throughout.

**Run order (cluster, from the repo root, kernel pd39)**
1. Part 0–1 (`ks-00` … `ks-11`): config, critical epochs → `output/cis25/ks_tracks/ks_track_epochs.fits`.
2. Part 2 (`ks-20`, `BUILD_PLAN=True` once): extraction plan → `sbatch` command printed → run it.
3. Part 3 (`ks-30`, `ks-31`): measurements → `ks_track_measurements.fits` + anchor QC (files added since the
   cached run — e.g. new `end` snapshots — are measured incrementally after their reduced files exist).
4. Part 4 (`ks-40` … `ks-60`): join, SIMBA-only diagnostics vs the reference relation `RELATIONS_DRAW` = Bigiel+08
   (an H$_2$ relation, ±0.20 dex; K98 thin for orientation) (anchors →
   `output/cis25/plots/ks_tracks/ks_anchors_R50_H2.png`; every critical point → `ks_stages_R50_H2.png`,
   all-z `ks_stages_allz_R50_H2.png`; counts → `ks_stage_regions.csv`), binned average-track figures (8 schemes, R$_{50}$(H$_2$)
   only, stages SFT → QT → end → `ks_tracks_binned_<scheme>_R50_H2.png` + one all-anchors panel per scheme
   `ks_tracks_binned_<scheme>_allz_R50_H2.png`), t$_{\rm dep}$ clock, Figure 3 fair
   comparison (Δ vs t$_{\rm dep}$; H$_2$ relation B08; total neutral gas vs K98 → `ks_fair_comparison_R50_H2.png` +
   `ks_fair_comparison.csv`), summary tables.
5. Part 5 (`ks-70` … `ks-78`): the three **KS regions** of the track end followed through their histories — continuous
   per-snapshot tracks (`ks_track_histories.fits`: history + BH history + caesar rotation via the progenitor index; Part 5a
   rebuilds the progenitor index per anchor like Part 1) → quench timing, AGN class / $f_{\rm Edd}$, the
   $M_{\rm dust}/M_\star$–age plane, $\kappa_{\rm rot}$, molecular content and extent on the clock $t - t_{\rm QT}$
   (`ks_regions_<what>_R50_H2.png`, `ks_region_properties.csv`); `ks-78` = presentation versions `ks_pres_{agn,dust,kinematics}`.

Local tests of the measurement code: `python -m pytest tests/test_ks_tracks.py -v`.

In [ ]:
# ── Part 0 — configuration ────────────────────────────────────────────────────
import os
import gc
import glob
import shutil
import tempfile
import warnings
import numpy as np
import h5py
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from astropy.io import fits
from astropy.table import Table, join, vstack
from astropy.cosmology import Planck15 as COSMO      # same as the quenching machinery / histories

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory
from simbanator.analysis.quenching import find_quenching_times
import ks_tracks_lib as kl                           # repo-root module (pure numpy; tests/test_ks_tracks.py)

# simbanator.analysis sets a huge global font on import — reset to something sane for these figures
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 12,
                     "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 8.5})

# ── simulation ────────────────────────────────────────────────────────────────
SIM_NAME = "cis25"        # SIMBA high-res 25 Mpc/h box (m25n512); must exist in ~/.simbanator/config.json
try:
    sim = Simulation(SIM_NAME)
except KeyError as e:
    raise KeyError(
        f"'{SIM_NAME}' is not registered in ~/.simbanator/config.json on this machine.\n"
        "Register it once (adjust paths to where the 25 Mpc snapshots+catalogs live):\n"
        "  from simbanator.io.config import add_simulation\n"
        "  add_simulation('cis25', data_dir='<...>/SIMBA_25',\n"
        "                 catalog_dir='<...>/SIMBA_25/Groups',\n"
        "                 file_format='m25n512_{snap:03d}.hdf5')\n"
        "then add \"snap_z_map\": \"zsnap_map_caesar_box100.txt\" to that entry."
    ) from e
if sim.scale_factors is None:
    raise ValueError(f"'{SIM_NAME}' config has no snap_z_map")
PARTICLE_PREFIX = sim.file_format.split("_{")[0]     # 'm25n512' — reduced-file prefix (job derives the same)

# ── the m25 sample (must match powderday_flux_quenched_m25.ipynb Part 0) ──────
TARGET_REDSHIFTS = [0.3, 0.5, 0.7, 0.85, 1.0, 1.15, 1.3, 1.5, 1.8, 2.0]
MASS_FLOOR, PASSIVE_FACTOR = 10.0, 0.2               # reference only (selection already made)

# ── paths ─────────────────────────────────────────────────────────────────────
OUT      = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR   = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
SELECTION_FITS = os.path.join(TABLEDIR, "powderday_quenched_selection.fits")
KSDIR    = os.path.join(OUT, "ks_tracks")
FIGDIR   = os.path.join(OUT, "plots", "ks_tracks")
PLAN_DIR = os.path.join(SFHDIR, "prof_kstracks")
PLAN_PATH = os.path.join(PLAN_DIR, "dust_profile_plan_kstracks.hdf5")
REDUCED_DIR = os.path.join(OUT, "reduced_particles")
REDUCED_PREFIX = PARTICLE_PREFIX
OBS_CSV  = os.path.join(os.getcwd(), "obs_data", "almac11", "ks_table.csv")
EPOCHS_FITS = os.path.join(KSDIR, "ks_track_epochs.fits")
MEAS_FITS   = os.path.join(KSDIR, "ks_track_measurements.fits")
TRACKS_FITS = os.path.join(KSDIR, "ks_tracks.fits")
for _d in (KSDIR, FIGDIR):
    os.makedirs(_d, exist_ok=True)

# ── stages / apertures / floors ───────────────────────────────────────────────
STAGES_KS   = list(kl.STAGES_KS)          # measured: sf_peak, ssfr_min, sft, qt, post_quench, gas_min, anchor, end
STAGES_PLOT = list(kl.STAGES_PLOT)        # time order (summary / t_dep clock): sf_peak, sft, qt, post_quench, gas_min, anchor
STAGES_DRAW = ["sft", "qt", "end"]        # drawn on the KS plane: quench start -> quench end -> track end (nothing else required)
END_STAGES  = ("anchor", "end")           # endpoint stages: big black-edged marker, SF controls shown in their Figure-0 panels
FH2_MIN_END = kl.FH2_MIN_END              # 1e-4: `end` = the anchor when catalogue M_H2/M* > this, else the last snapshot above it
RELATIONS_DRAW = ("B08",)                # THE reference relation (with its band): region colouring, ksreg bins, Figure 0/3 — Bigiel+08 is an H2 relation
RELATIONS_FAINT = ("K98",)               # drawn thin, no band, for orientation only (a total-gas relation; all kl.RELATIONS stay in the summary offsets)
STAGE_LABEL = {"sf_peak": "sSFR peak", "ssfr_min": "sSFR min", "sft": "SFT (quench start)",
               "qt": "QT (quench end)", "post_quench": "post-quench", "gas_min": r"H$_2$ trough",
               "anchor": "anchor", "end": "track end"}
END_DEF = r"track end = the anchor, or the last snapshot with $M_{\rm H_2}/M_\star > 10^{-4}$ when the anchor holds no H$_2$"
STAGE_MARKER = {"sf_peak": "*", "ssfr_min": "x", "sft": "^", "qt": "s", "post_quench": "D",
                "gas_min": "v", "anchor": "o", "end": "o"}
STAGE_COLOR = {"sf_peak": "#1b9e77", "sft": "#e6ab02", "qt": "#d95f02", "post_quench": "#a6761d",
               "gas_min": "#7570b3", "anchor": "#1f1f1f", "end": "#1f1f1f", "ssfr_min": "0.5"}
FIXED_AP_KPC = tuple(kl.FIXED_AP_KPC)     # 1, 3.162, 10 kpc  (labels ap1kpc / ap3kpc / ap10kpc)
AP_LABELS    = list(kl.AP_LABELS)         # + R50_H2 / R50_star / R50_SFR
AP_TITLE = {"ap1kpc": r"$r<1$ kpc", "ap3kpc": r"$r<3.2$ kpc", "ap10kpc": r"$r<10$ kpc",
            "R50_H2": r"$r<R_{50}({\rm H_2})$", "R50_star": r"$r<R_{50}(\star)$",
            "R50_SFR": r"$r<R_{50}({\rm SFR})$"}
FIDUCIAL_AP  = "R50_H2"                   # observed convention: 0.5 M / (pi R_CO^2)
FIDUCIAL_SFR = "sfr100"                   # archaeological 100 Myr window (slist)
MEMBER_ONLY  = True                       # CAESAR glist / slist particles only
NGAS_MIN, NH2_MIN, NSTAR_MIN = 10, 5, 10  # particle floors (aperture gas / R50_H2 / R50_star)
SFR_WINDOWS_MYR = (25.0, 100.0)
HE_FACTOR = kl.HE_FACTOR                  # 1.36: simulated H2 is hydrogen-only, observed alpha_CO includes He
INCLUDE_SF_CONTROL = True                 # SF partners at their anchor as a reference cloud
Z_OBS = 0.37                              # ALMA-C11 median redshift (Tacconi+18 MS t_dep evaluated here)
A_TO_T = kl.make_a_to_t(COSMO)            # star formation scale factor -> cosmic time [Gyr]

# ── gates (heavy or one-off steps; caches are reused when the flags are False) ──
BUILD_PLAN             = False            # Part 2: write the SLURM extraction plan
OVERWRITE_EPOCHS       = False            # Part 1 cache
OVERWRITE_MEASUREMENTS = False            # Part 3 cache


def _ztag(z):
    return ("z%g" % z).replace(".", "p")


def _s(col):
    """FITS string column -> stripped str array (bytes on some numpy/astropy combos)."""
    return np.char.strip(np.asarray(col).astype(str))


def write_table(tab, path):
    """astropy Table -> FITS via a temp file in the same directory (atomic; gvfs-safe pattern)."""
    d = os.path.dirname(path)
    os.makedirs(d, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", suffix=".fits", dir=d)
    os.close(fd)
    tab.write(tmp, overwrite=True)
    os.replace(tmp, path)
    return path


print(f"sim={sim.name}  prefix={PARTICLE_PREFIX}  anchors z={TARGET_REDSHIFTS}")
print(f"fiducial: aperture={FIDUCIAL_AP}  SFR={FIDUCIAL_SFR}  member_only={MEMBER_ONLY}  He x{HE_FACTOR}")
print("stages:", STAGES_KS, "| drawn on the KS plane:", STAGES_DRAW, "| relation(s):", RELATIONS_DRAW)
print("selection:", SELECTION_FITS, "->", "ok" if os.path.exists(SELECTION_FITS) else "MISSING")
print("observed:", OBS_CSV, "->", "ok" if os.path.exists(OBS_CSV) else "MISSING (git pull?)")

In [ ]:
# ── anchor table (snapshot nearest each target z; per-anchor history / progenitor products) ──
_sall, _zall = [], []
for _s_ in range(0, 152):
    try:
        _zv = float(sim.get_z_from_snap(_s_))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s_); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)

ANCHORS = {}
for _zt in TARGET_REDSHIFTS:
    _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
    _tag = _ztag(_zt)
    ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap, z=float(sim.get_z_from_snap(_snap)),
                        prog_file=f"progenitors_anchor_{_tag}.fits",
                        hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"))

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'hist':>6s} {'prog':>6s}")
for _zt, A in ANCHORS.items():
    _pf = os.path.join(OUT, "progenitors", A["prog_file"])
    print(f"{_zt:6.2f} {A['snap']:5d} {A['z']:7.3f} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(_pf) else '--':>6s}")

In [ ]:
# ── loaders (verbatim from powderday_flux_quenched_m25 Parts 0b/1): row 0 = the anchor epoch ──
def load_selection():
    """SELECTION_FITS (written by the m25 notebook Part 3) -> (table, snap array, gal_id array)."""
    sel = Table.read(SELECTION_FITS)
    return sel, np.asarray(sel["snap"], int), np.asarray(sel["gal_id"], int)


def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H


def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows
    (walks tree_data/progen_galaxy_star through the progen_links sidecars; cwd must be the repo root)."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP
    gc.collect()
    return M

## Part 1 — critical epochs of every quenched galaxy

One row per (anchor, galaxy, stage) with the snapshot + catalogue index of the progenitor at that
epoch (`gx`, −1 when the progenitor chain is broken or the stage lies beyond the anchor), the
epoch times, and the catalogue-level history values at that row (for the Part 3 cross-check).
`t_qt` is recomputed with the same finder and the same history as the selection table and must
agree exactly. Star-forming partners (`pop = SF`) get an `anchor` row only (they have no history).

In [ ]:
# ── Part 1 — epochs table (cached) ──
_HIST_KEYS = {"mstar_cat": "masses.stellar", "sfr_cat": "sfr", "mh2_cat": "masses.H2",
              "mgas_cat": "masses.gas", "mdust_cat": "masses.dust",
              "r50star_cat": "radii.stellar_half_mass", "r50gas_cat": "radii.gas_half_mass",
              "ngas_cat": "ngas", "nstar_cat": "nstar"}

EPOCHS = None
if os.path.exists(EPOCHS_FITS) and not OVERWRITE_EPOCHS:
    EPOCHS = Table.read(EPOCHS_FITS)
    print(f"cached ({len(EPOCHS)} rows) -> {EPOCHS_FITS}   (OVERWRITE_EPOCHS=True rebuilds)")
    if "end" not in set(_s(EPOCHS["stage"])):
        print("  the cached table predates the `end` stage -> Part 1 is rebuilt now (as with OVERWRITE_EPOCHS=True)")
        EPOCHS = None
if EPOCHS is None:
    SEL, SNAPS, IDS = load_selection()
    POP = _s(SEL["pop"])
    _rows, _tqt_check = [], []
    for _zt, A in ANCHORS.items():
        _inA = SNAPS == A["snap"]
        _q = SEL[_inA & (POP == "Q")]
        _sf = SEL[_inA & (POP == "SF")]
        if len(_q) == 0 and len(_sf) == 0:
            continue
        print(f"[{A['tag']}] snap {A['snap']} z={A['z']:.2f}: {len(_q)} Q, {len(_sf)} SF")
        if len(_q):
            H = load_anchor_history(A)
            _g2c = {int(g): j for j, g in enumerate(H["galaxy_ids"])}
            _missing = [int(g) for g in _q["gal_id"] if int(g) not in _g2c]
            if _missing:
                raise KeyError(f"[{A['tag']}] {len(_missing)} selected galaxies absent from the history: {_missing[:10]}")
            _cols = np.array([_g2c[int(g)] for g in _q["gal_id"]], int)
            _recs = kl.build_stage_records(H["P"], H["t_cosmic_yr"], H["redshift"], H["galaxy_ids"],
                                           _cols, find_quenching_times,
                                           age_of_z_gyr=lambda z: COSMO.age(z).value, fh2_min=FH2_MIN_END)
            PIDX = build_prog_index(A, H["galaxy_ids"], H["snaps_arr"])
            for _rec, _qrow in zip(_recs, _q):
                _tqt_check.append((float(_qrow["t_qt"]), _rec["t_qt"]))
                for _st in STAGES_KS:
                    _row = int(_rec["row_%s" % _st])
                    _tdef = _rec["t_%s" % _st]
                    if _row < 0 and not np.isfinite(_tdef):
                        continue                      # stage undefined for this galaxy
                    _gx = PIDX[_row, _rec["col"]] if _row >= 0 else np.nan
                    _t = float(H["t_cosmic_yr"][_row]) / 1e9 if _row >= 0 else np.nan
                    _d = dict(anchor_z=float(_zt), anchor_snap=int(A["snap"]), gal_id=int(_rec["gid"]),
                              pop="Q", stage=_st, row=_row,
                              snap=int(H["snaps_arr"][_row]) if _row >= 0 else -1,
                              gx=int(_gx) if np.isfinite(_gx) else -1,
                              z=float(H["redshift"][_row]) if _row >= 0 else np.nan,
                              t_stage_gyr=_t, t_def_gyr=_tdef / 1e9 if np.isfinite(_tdef) else np.nan,
                              dt_from_qt_gyr=(_t - _rec["t_qt"] / 1e9) if (np.isfinite(_t) and np.isfinite(_rec["t_qt"])) else np.nan,
                              t_sft_gyr=_rec["t_sft"] / 1e9, t_qt_gyr=_rec["t_qt"] / 1e9,
                              tau_q_gyr=_rec["tau_q"] / 1e9, z_qt=_rec["z_qt"], n_events=int(_rec["n_events"]),
                              end_is_anchor=bool(_rec["end_is_anchor"]), fh2_anchor=float(_rec["fh2_anchor"]),
                              agn_class=str(_s(np.array([_qrow["agn_class"]]))[0]),
                              xstr_quench=float(_qrow["xstr_quench"]),
                              log_mstar_anchor=float(_qrow["log_mstar"]))
                    for _k, _pk in _HIST_KEYS.items():
                        _d[_k] = float(H["P"][_pk][_row, _rec["col"]]) if (_row >= 0 and _pk in H["P"]) else np.nan
                    _rows.append(_d)
            del H, PIDX
            gc.collect()
        if INCLUDE_SF_CONTROL and len(_sf):
            _tA = float(COSMO.age(A["z"]).value)
            for _r in _sf:
                _ms = 10.0 ** float(_r["log_mstar"])
                _d = dict(anchor_z=float(_zt), anchor_snap=int(A["snap"]), gal_id=int(_r["gal_id"]),
                          pop="SF", stage="anchor", row=0, snap=int(A["snap"]), gx=int(_r["gal_id"]),
                          z=float(A["z"]), t_stage_gyr=_tA, t_def_gyr=_tA, dt_from_qt_gyr=np.nan,
                          t_sft_gyr=np.nan, t_qt_gyr=np.nan, tau_q_gyr=np.nan, z_qt=np.nan, n_events=0,
                          end_is_anchor=True, fh2_anchor=np.nan, agn_class="SF", xstr_quench=np.nan, log_mstar_anchor=float(_r["log_mstar"]),
                          mstar_cat=_ms, sfr_cat=float(_r["ssfr"]) * _ms, mh2_cat=float(_r["mh2"]),
                          mgas_cat=np.nan, mdust_cat=float(_r["mdust"]), r50star_cat=np.nan,
                          r50gas_cat=np.nan, ngas_cat=float(_r["ngas"]), nstar_cat=float(_r["nstar"]))
                _rows.append(_d)
    EPOCHS = Table(rows=_rows)
    # the selection's t_qt came from the same finder on the same history: must agree exactly
    _c = np.array(_tqt_check, float)
    _both = np.isfinite(_c).all(axis=1)
    _dmax = np.abs(_c[_both, 0] - _c[_both, 1]).max() if _both.any() else 0.0
    _nfin = int((np.isfinite(_c[:, 0]) != np.isfinite(_c[:, 1])).sum())
    print(f"t_qt check vs selection: {int(_both.sum())} finite pairs, max |dt| = {_dmax:.3g} yr, "
          f"{_nfin} finiteness mismatches")
    if _dmax > 1e3 or _nfin:
        warnings.warn("recomputed t_qt disagrees with powderday_quenched_selection.fits — "
                      "history files or find_quenching_times changed since the selection was made")
    write_table(EPOCHS, EPOCHS_FITS)
    print(f"wrote {len(EPOCHS)} rows -> {EPOCHS_FITS}")

_st_col, _pop_col = _s(EPOCHS["stage"]), _s(EPOCHS["pop"])
print("rows per stage (Q):", {st: int(((_st_col == st) & (_pop_col == "Q")).sum()) for st in STAGES_KS})
print("SF control rows:", int((_pop_col == "SF").sum()),
      "| Q stage rows with a tracked progenitor (gx>=0):",
      int(((_pop_col == "Q") & (np.asarray(EPOCHS["gx"]) >= 0)).sum()))
_endQ = (_st_col == "end") & (_pop_col == "Q")
_eia = np.asarray(EPOCHS["end_is_anchor"], bool)
print(f"end stage (Q): {int(_endQ.sum())} rows -> {int((_endQ & _eia).sum())} = the anchor (catalogue M_H2/M* > {FH2_MIN_END:g}), "
      f"{int((_endQ & ~_eia & (np.asarray(EPOCHS['gx']) >= 0)).sum())} fall back to the last snapshot with M_H2/M* > {FH2_MIN_END:g}, "
      f"{int((_endQ & ~_eia & (np.asarray(EPOCHS['gx']) < 0)).sum())} never above it (no end row)")

In [ ]:
# ── Part 1 QC — stage availability funnel, coincident epochs, time-since-QT per stage ──
_stg, _pop = _s(EPOCHS["stage"]), _s(EPOCHS["pop"])
_gx = np.asarray(EPOCHS["gx"], int)
_isQ = _pop == "Q"
print(f"{'anchor':>7s} " + " ".join(f"{st[:9]:>11s}" for st in STAGES_KS) + "   (defined/tracked)")
for _zt, A in ANCHORS.items():
    _inA = _isQ & (np.asarray(EPOCHS["anchor_snap"]) == A["snap"])
    if not _inA.any():
        continue
    _cells = []
    for st in STAGES_KS:
        _m = _inA & (_stg == st)
        _cells.append(f"{int(_m.sum()):4d}/{int((_m & (_gx >= 0)).sum()):<4d}")
    print(f"{_zt:7.2f} " + " ".join(f"{c:>11s}" for c in _cells))

# coincident epochs: several stages of one galaxy on the same (snap, gx) -> one reduced file serves them
_keyQ = [(int(a), int(g), int(s), int(x)) for a, g, s, x in
         zip(EPOCHS["anchor_snap"][_isQ & (_gx >= 0)], EPOCHS["gal_id"][_isQ & (_gx >= 0)],
             EPOCHS["snap"][_isQ & (_gx >= 0)], _gx[_isQ & (_gx >= 0)])]
_uniq = len(set(_keyQ))
print(f"\nQ epoch rows with a file: {len(_keyQ)} -> {_uniq} distinct (galaxy, snap) -> "
      f"{len(set((s, x) for _, _, s, x in _keyQ))} distinct reduced files (snap, gx)")
_beyond = _isQ & (_stg == "post_quench") & (_gx < 0) & np.isfinite(np.asarray(EPOCHS["t_def_gyr"]))
print(f"post_quench beyond the anchor (not observable in the history): {int(_beyond.sum())} galaxies")

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for st in ("sft", "qt", "post_quench", "gas_min", "anchor"):
    _m = _isQ & (_stg == st) & (_gx >= 0) & np.isfinite(np.asarray(EPOCHS["dt_from_qt_gyr"]))
    if _m.sum():
        axs[0].hist(np.asarray(EPOCHS["dt_from_qt_gyr"])[_m], bins=np.linspace(-4, 8, 49),
                    histtype="step", lw=1.6, color=STAGE_COLOR[st], label=f"{STAGE_LABEL[st]} (N={int(_m.sum())})")
axs[0].axvline(0, color="0.5", lw=0.8)
axs[0].set_xlabel(r"$t_{\rm stage} - t_{\rm QT}$ [Gyr]"); axs[0].set_ylabel("galaxies"); axs[0].legend()
axs[0].set_title("(a) time of each stage relative to QT", loc="left")
_m = _isQ & (_stg == "anchor")
axs[1].scatter(np.asarray(EPOCHS["z"])[_m], np.asarray(EPOCHS["z_qt"])[_m], s=12, c="0.3")
axs[1].plot([0, 3], [0, 3], color="0.6", lw=0.8)
axs[1].set_xlabel("anchor z"); axs[1].set_ylabel(r"$z_{\rm QT}$"); axs[1].set_title("(b) when they quenched", loc="left")
plt.tight_layout(); fig.savefig(os.path.join(FIGDIR, "ks_epochs_qc.png"), dpi=150); plt.show()

## Part 2 — extraction plan for the reduced particle files

The unique (snapshot, catalogue index) pairs of Part 1 go into ONE plan file in the schema
`build_reduced_particles_job.py` reads (`sim_name` attr + `entry_gx` / `entry_snap`), plus a
provenance group the job ignores. The job is idempotent: files that already exist (e.g. the
box-comparison campaign at snaps 105/134) are only **backfilled** with the new `tform` field
(star formation epochs → archaeological SFR); nothing is re-extracted.
Set `BUILD_PLAN=True`, run this cell once, then submit on the cluster (command printed below).

In [ ]:
# ── Part 2 — plan (gated) ──
_ok = (np.asarray(EPOCHS["gx"], int) >= 0) & (np.asarray(EPOCHS["snap"], int) > 0)
_pairs = sorted(set((int(s), int(g)) for s, g in zip(EPOCHS["snap"][_ok], EPOCHS["gx"][_ok])))
_snaps_u = sorted(set(s for s, _ in _pairs))
_have = [os.path.exists(kl.reduced_path(REDUCED_DIR, REDUCED_PREFIX, s, g)) for s, g in _pairs]
print(f"{len(_pairs)} unique (snap, gx) over {len(_snaps_u)} snapshots "
      f"({min(_snaps_u)}..{max(_snaps_u)}); reduced files already on disk: {sum(_have)}")

# plan path as the script's first ARGUMENT: sbatch hands script arguments to the job verbatim, whereas a
# `DUST_PLAN=... sbatch` env prefix only reaches the job when it sits on the same command line
# (the 2026-08-27 arrays 13356106-154 died on the script's guard with the variable unset)
SBATCH_CMD = ("cd /mnt/home/glorenzon/analize_simba_cgm && mkdir -p logs && "
              "sbatch --array=0-15%4 submit_reduced_particles.sh "
              + os.path.relpath(PLAN_PATH, "/mnt/home/glorenzon/analize_simba_cgm"))
if BUILD_PLAN:
    os.makedirs(PLAN_DIR, exist_ok=True)
    _tmp = PLAN_PATH + ".part"
    with h5py.File(_tmp, "w") as f:
        f.attrs["sim_name"] = SIM_NAME
        f.attrs["note"] = "ks_tracks_quenched_m25.ipynb: quenched sample at its critical epochs (+ SF controls at the anchor)"
        f.attrs["rmax_kpc"] = 100.0
        f.attrs["stages"] = ",".join(STAGES_KS)
        f.attrs["selection_fits"] = SELECTION_FITS
        f.attrs["n_epochs"] = int(_ok.sum())
        f.create_dataset("entry_gx", data=np.array([g for _, g in _pairs], np.int64))
        f.create_dataset("entry_snap", data=np.array([s for s, _ in _pairs], np.int32))
        m = f.create_group("map")                        # provenance, ignored by the job
        _E = EPOCHS[_ok]
        for c in ("anchor_snap", "gal_id", "snap", "gx"):
            m.create_dataset(c, data=np.asarray(_E[c], np.int64))
        for c in ("pop", "stage"):
            m.create_dataset(c, data=np.asarray(_s(_E[c]), dtype=object), dtype=h5py.string_dtype())
    os.replace(_tmp, PLAN_PATH)
    print(f"wrote plan -> {PLAN_PATH}")
    print("submit on the cluster (16 array tasks over snapshots, throttled to 4 concurrent):\n  " + SBATCH_CMD)
elif os.path.exists(PLAN_PATH):
    with h5py.File(PLAN_PATH, "r") as f:
        _pp = set(zip(f["entry_snap"][:].astype(int), f["entry_gx"][:].astype(int)))
    _new = set(_pairs) - _pp
    print(f"existing plan: {len(_pp)} entries; {len(_new)} current pairs NOT in it "
          + ("(BUILD_PLAN=True to refresh)" if _new else "(up to date)"))
    print("submit / resubmit with:\n  " + SBATCH_CMD)
else:
    print("no plan yet -> set BUILD_PLAN=True and re-run this cell")

### Cluster steps between Part 2 and Part 3

```bash
cd /mnt/home/glorenzon/analize_simba_cgm && mkdir -p logs
sbatch --array=0-15%4 submit_reduced_particles.sh output/cis25/caesar_sfh/prof_kstracks/dust_profile_plan_kstracks.hdf5
```
* The plan is the script's first argument (`sbatch` forwards script arguments to the job
  verbatim). The older `DUST_PLAN=<plan> sbatch ...` env form is still accepted, but only when the
  assignment is on the *same* line as `sbatch` — on its own line it is an un-exported shell variable
  and the job dies at the `DUST_PLAN` guard (`logs/reduced_*.err`).
* Re-running that same command is also how a **recipe refresh** is applied: files whose `h2_recipe` attr is not the current `build_profiles_job.H2_RECIPE` get `m_HI`/`m_H2` recomputed at the stored particle indices (no geometry redo). The 2026-08-27 files were built with the old `m_H·nh·fH2` split (up to 97% of the H2 missing in star-forming gas of massive galaxies) and must be refreshed this way before Part 3.
* 16 array tasks split the plan's snapshots (newest first); `%4` keeps at most four 32 GB tasks
  running at once — SLURM here schedules on cores only, so un-throttled memory-hungry arrays
  OOM-kill each other on the shared phi nodes.
* Progress: `grep -h "snap" logs/reduced_*.out` (per snapshot: N planned → extracted / backfilled /
  already complete); `grep -h done: logs/reduced_*.out` when finished. Re-submitting the same command
  fills any gaps (idempotent).
* Then run Part 3. Files still missing are reported there (job not finished / progenitor without
  a catalogue entry).

## Part 3 — Σ$_{\rm H_2}$ and SFR from the member particles at every epoch

Each reduced file → face-on cylindrical radii in the stored stellar principal frame
(`pos @ evecs`, columns = axes) → CAESAR members only → in every aperture (three fixed rungs and the
three half-mass radii): H$_2$/HI/gas/dust mass, instantaneous gas SFR, archaeological SFR over
25 / 100 Myr from the star formation epochs, particle counts. One row per (snap, gx, aperture);
member totals and the half-mass radii are repeated on every row of a file.

In [ ]:
# ── Part 3 — measurements (cached) ──
_KEYS = {"gas": ("pos", "m_gas", "m_H2", "m_HI", "m_dust", "sfr", "member"),
         "star": ("pos", "m_star", "member", "tform")}


def _measure_pairs(pairs):
    """measure_ks over the reduced files of (snap, gx) `pairs` -> (rows, n_missing, bad paths, n without tform)."""
    rows, bad = [], []
    n_missing = n_notform = 0
    for k, (snap, gx) in enumerate(pairs):
        p = kl.reduced_path(REDUCED_DIR, REDUCED_PREFIX, snap, gx)
        red = kl.load_reduced(p, keys=_KEYS, bad=bad)
        if red is None:
            n_missing += 1
            continue
        zs = float(red["redshift"])
        t_obs = float(COSMO.age(zs).value)
        rr = kl.measure_ks(red, t_obs, A_TO_T, fixed_kpc=FIXED_AP_KPC, member_only=MEMBER_ONLY,
                           ngas_min=NGAS_MIN, nh2_min=NH2_MIN, nstar_min=NSTAR_MIN, sfr_windows=SFR_WINDOWS_MYR)
        if not rr[0]["has_tform"]:
            n_notform += 1
        for r in rr:
            r.update(snap=int(snap), gx=int(gx), z_file=zs, t_obs_gyr=t_obs)
            rows.append(r)
        if (k + 1) % 200 == 0:
            print(f"  {k + 1}/{len(pairs)} files")
    return rows, n_missing, bad, n_notform


MEAS = None
if os.path.exists(MEAS_FITS) and not OVERWRITE_MEASUREMENTS:
    MEAS = Table.read(MEAS_FITS)
    _have = set(zip(np.asarray(MEAS["snap"], int), np.asarray(MEAS["gx"], int)))
    print(f"cached ({len(MEAS)} rows, {len(_have)} files) -> {MEAS_FITS}   (OVERWRITE_MEASUREMENTS=True rebuilds)")
    _todo = [p for p in _pairs if p not in _have]       # pairs added since the cached run (e.g. `end` snapshots): incremental
    if _todo:
        _rows, _n_missing, _bad, _n_notform = _measure_pairs(_todo)
        if _rows:
            MEAS = vstack([MEAS, Table(rows=_rows)])
            write_table(MEAS, MEAS_FITS)
        print(f"  {len(_todo)} new (snap, gx) pairs: measured {len(_todo) - _n_missing - len(_bad)}, "
              f"missing files {_n_missing} (run the sbatch command of Part 2 first), corrupt {len(_bad)}, without tform {_n_notform}")
if MEAS is None:
    _rows, _n_missing, _bad, _n_notform = _measure_pairs(_pairs)
    MEAS = Table(rows=_rows)
    write_table(MEAS, MEAS_FITS)
    print(f"measured {len(_pairs) - _n_missing - len(_bad)}/{len(_pairs)} files -> {MEAS_FITS}")
    print(f"  missing files: {_n_missing} (job not finished?)   corrupt: {len(_bad)}   "
          f"without tform (old files not yet backfilled): {_n_notform}")

_apl = _s(MEAS["ap_label"])
for _lab in AP_LABELS:
    _m = _apl == _lab
    _fin = _m & np.isfinite(np.asarray(MEAS["m_H2"], float)) & (np.asarray(MEAS["m_H2"], float) > 0)
    print(f"  {_lab:9s}: {int(_m.sum())} rows, Sigma_H2 measurable (n_gas>={NGAS_MIN}, M_H2>0): {int(_fin.sum())}, "
          f"SFR100=0: {int((_m & (np.asarray(MEAS['sfr100'], float) == 0)).sum())}")

In [ ]:
# ── Part 3 QC — member totals at the anchor vs the catalogue (progenitor index + member flag + units) ──
_E = EPOCHS[(_s(EPOCHS["stage"]) == "anchor") & (np.asarray(EPOCHS["gx"]) >= 0)]
_M = MEAS[_s(MEAS["ap_label"]) == "R50_H2"]           # any label carries the file totals
_J = join(_E, _M, keys=("snap", "gx"), join_type="inner")
_popJ = _s(_J["pop"])
print(f"anchor rows joined: {len(_J)} ({int((_popJ == 'Q').sum())} Q, {int((_popJ == 'SF').sum())} SF)")


def _ratio_stats(name, num, den):
    num, den = np.asarray(num, float), np.asarray(den, float)
    ok = np.isfinite(num) & np.isfinite(den) & (den > 0) & (num > 0)
    if not ok.any():
        print(f"  {name:28s}: no finite pairs"); return None
    r = np.log10(num[ok] / den[ok])
    print(f"  {name:28s}: N={int(ok.sum()):4d}  median log(sim/cat) = {np.median(r):+.3f}  "
          f"16-84 = [{np.percentile(r, 16):+.3f}, {np.percentile(r, 84):+.3f}]")
    return r


print("member totals vs catalogue (expect ~0 for masses; radii differ by definition: face-on 2D vs 3D):")
_rm = _ratio_stats("M_H2 (member sum / cat)", _J["m_H2_tot"], _J["mh2_cat"])
_rs = _ratio_stats("M* (member sum / cat)", _J["m_star_tot"], _J["mstar_cat"])
_rf = _ratio_stats("SFR_inst (member / cat)", _J["sfr_inst_tot"], _J["sfr_cat"])
_ra = _ratio_stats("SFR100 (archaeol. / cat inst)", _J["sfr100_tot"], _J["sfr_cat"])
_rr = _ratio_stats("R50* face-on / cat half-mass", _J["r50_star"], _J["r50star_cat"])

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (xk, yk, lab) in zip(axs, (("mh2_cat", "m_H2_tot", r"$M_{\rm H_2}$ [M$_\odot$]"),
                                   ("mstar_cat", "m_star_tot", r"$M_\star$ [M$_\odot$]"),
                                   ("sfr_cat", "sfr_inst_tot", r"SFR$_{\rm inst}$ [M$_\odot$ yr$^{-1}$]"))):
    x, y = np.asarray(_J[xk], float), np.asarray(_J[yk], float)
    ok = (x > 0) & (y > 0)
    ax.scatter(x[ok & (_popJ == "SF")], y[ok & (_popJ == "SF")], s=8, c="0.6", label="SF control")
    ax.scatter(x[ok & (_popJ == "Q")], y[ok & (_popJ == "Q")], s=10, c="#d95f02", label="Q")
    lo, hi = np.nanmin(np.r_[x[ok], y[ok]]), np.nanmax(np.r_[x[ok], y[ok]])
    ax.plot([lo, hi], [lo, hi], color="0.4", lw=0.8)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("catalogue " + lab); ax.set_ylabel("member sum " + lab)
axs[0].legend(); axs[0].set_title("anchor epoch: member particles reproduce the catalogue?", loc="left")
plt.tight_layout(); fig.savefig(os.path.join(FIGDIR, "ks_anchor_qc.png"), dpi=150); plt.show()

## Part 4 — the Kennicutt–Schmidt plane

Join epochs × apertures, scale to the observational conventions (`ks_tracks_lib.ks_columns`), overlay
the observed ALMA-C11 galaxies and the Kennicutt 98 relation, and draw the **average tracks**:
one figure per binning scheme in the fiducial aperture R$_{50}$(H$_2$) only (a fixed aperture is a
different Σ definition from the observed one, so the 3.2 / 10 kpc rows are summarised, not drawn), three
panels (anchors z ≤ 0.5 = the ALMA-C11 range, 0.7–1, 1.15–2), and in every panel the per-stage median
track of each **bin** over the quench phase (SFT → QT → track end; no scatter bars). The track end is the anchor,
or — when the anchor holds no measurable H$_2$ (catalogue M$_{\rm H_2}$/M$_\star$ ≤ 10$^{-4}$) — the last snapshot with
M$_{\rm H_2}$/M$_\star$ > 10$^{-4}$ (`end` rows of Part 1); no other stage is required of a galaxy.
Individual galaxies are not drawn on the track figures. Eight binning schemes, all evaluated at the galaxy's
anchor: catalogue SFR, stellar mass, dust fraction M$_{\rm dust}$/M$_\star$, dust surface density
0.5 M$_{\rm dust}$/πR$_{50\star}^2$, the galaxy's own Σ$_{\rm H_2}$ (the plane's x-coordinate at the anchor),
the molecular fraction M$_{\rm H_2}$/M$_\star$ (terciles of the full quenched sample), the **region where the
track ends with respect to the reference relation** (`RELATIONS_DRAW` = Bigiel+08, an H$_2$ relation: Δ = log Σ$_{\rm SFR}$ −
B08(Σ$_{\rm H_2}$) below / within / above its ±0.20 dex band; K98 is drawn thin for orientation only) — where do the galaxies that end below, on, or above the relation come from?), and the AGN coupling
class during the quenching phase.
Each scheme gives two figures: the three anchor-redshift panels and one panel with **all anchors together** (`_allz`, same bin edges).
**Figure 0** first shows the SIMBA galaxies alone against the reference relation, no tracks, no observed points, every galaxy coloured
by its region (below / on / above the band) *at that point*: (0a) the quenched and star-forming anchors
→ `ks_anchors_R50_H2.png`; (0b) the quenched galaxies at every critical point of the quench phase (rows
SFT → QT → end × the three anchor-redshift ranges) → `ks_stages_R50_H2.png`; (0c) the same with
all anchor redshifts together → `ks_stages_allz_R50_H2.png`. Counts per (stage, z range) → `ks_stage_regions.csv`
— is there a point where enough galaxies sit below the relation, like the observed ones?
Track figures → `ks_tracks_binned_<scheme>_R50_H2.png`; bin medians → `ks_binned_tracks.csv`.


In [ ]:
# ── Part 4a — join + KS columns (fiducial = archaeological 100 Myr; also 25 Myr and instantaneous) ──
_E = EPOCHS[np.asarray(EPOCHS["gx"]) >= 0]
TRACKS = join(_E, MEAS, keys=("snap", "gx"), join_type="inner")      # epoch rows x 6 aperture rows
for _k, _v in kl.ks_columns(TRACKS, "sfr100", "sfr100_tot", 100.0, HE_FACTOR).items():
    TRACKS[_k] = _v
for _suf, _key, _tot, _w in (("_sfr25", "sfr25", "sfr25_tot", 25.0), ("_inst", "sfr_inst", "sfr_inst_tot", 100.0)):
    for _k, _v in kl.ks_columns(TRACKS, _key, _tot, _w, HE_FACTOR, suffix=_suf).items():
        if _k != "logSigmaH2":
            TRACKS[_k] = _v
TRACKS["tdep_ms_gyr"] = kl.tdep_ms_gyr(np.asarray(TRACKS["z"], float))
# total NEUTRAL gas (HI + H2, x1.36 He) inside the aperture / its area — what the total-gas relations (K98, RK19) were calibrated on;
# on the R50_H2 rows the H2 half is 0.5 M_H2 by construction, the HI is the literal amount inside R50(H2)
with np.errstate(divide="ignore", invalid="ignore"):
    _mn = np.asarray(TRACKS["m_HI"], float) + np.asarray(TRACKS["m_H2"], float)
    _ar = np.asarray(TRACKS["area_kpc2"], float)
    TRACKS["logSigmaGas"] = np.where(np.isfinite(_mn) & (_mn > 0) & np.isfinite(_ar) & (_ar > 0), np.log10(HE_FACTOR * _mn / _ar / 1e6), np.nan)
TRACKS["gkey"] = np.array([f"{a}_{g}" for a, g in zip(TRACKS["anchor_snap"], TRACKS["gal_id"])])
write_table(TRACKS, TRACKS_FITS)
print(f"TRACKS: {len(TRACKS)} rows -> {TRACKS_FITS}")

# node export in the observed table's column names (for pilot_specphot/scripts/plot_ks.py overlays)
_ap, _stg, _pop = _s(TRACKS["ap_label"]), _s(TRACKS["stage"]), _s(TRACKS["pop"])
_nodes = pd.DataFrame(dict(
    id=np.asarray(TRACKS["gkey"]).astype(str), pop=_pop, stage=_stg, aperture=_ap,
    z=np.asarray(TRACKS["z"], float), anchor_z=np.asarray(TRACKS["anchor_z"], float),
    dt_from_qt_gyr=np.asarray(TRACKS["dt_from_qt_gyr"], float),
    agn_class=_s(TRACKS["agn_class"]), log_mstar=np.log10(np.asarray(TRACKS["mstar_cat"], float)),
    r_kpc=np.asarray(TRACKS["ap_kpc"], float),
    MH2_fid=HE_FACTOR * np.asarray(TRACKS["m_H2_tot"], float), SFR=np.asarray(TRACKS["sfr100_tot"], float),
    logSigmaH2=np.asarray(TRACKS["logSigmaH2"], float), logSigmaSFR=np.asarray(TRACKS["logSigmaSFR"], float),
    logSigmaGas=np.asarray(TRACKS["logSigmaGas"], float),
    is_ul=np.asarray(TRACKS["is_ul"], bool), tdep_Gyr=np.asarray(TRACKS["tdep_gyr"], float),
    logSigmaSFR_inst=np.asarray(TRACKS["logSigmaSFR_inst"], float), n_gas=np.asarray(TRACKS["n_gas"], int),
    n_H2=np.asarray(TRACKS["n_H2"], int)))
_nodes.to_csv(os.path.join(KSDIR, "ks_track_nodes.csv"), index=False)
print(f"nodes CSV -> {os.path.join(KSDIR, 'ks_track_nodes.csv')}")
_fid = (_ap == FIDUCIAL_AP) & (_pop == "Q")
print(f"fiducial ({FIDUCIAL_AP}, {FIDUCIAL_SFR}) Q rows: {int(_fid.sum())}; both Sigma finite: "
      f"{int((_fid & np.isfinite(TRACKS['logSigmaH2']) & np.isfinite(TRACKS['logSigmaSFR_obs'])).sum())}; "
      f"SFR censored (upper limits): {int((_fid & np.asarray(TRACKS['is_ul'], bool)).sum())}; "
      f"R50_H2 undefined: {int((_fid & ~np.isfinite(np.asarray(TRACKS['ap_kpc'], float))).sum())}")
print("funnel of the drawn stages (fiducial rows, Q): rows with a file / Sigma_H2 measurable / both Sigma finite & SFR uncensored")
for _st in STAGES_DRAW:
    _m = _fid & (_stg == _st)
    _mx = _m & np.isfinite(np.asarray(TRACKS["logSigmaH2"], float))
    _mb = _mx & np.isfinite(np.asarray(TRACKS["logSigmaSFR_obs"], float)) & ~np.asarray(TRACKS["is_ul"], bool)
    print(f"  {_st:12s} {int(_m.sum()):4d} / {int(_mx.sum()):4d} / {int(_mb.sum()):4d}"
          + (f"   (end == anchor for {int((_m & np.asarray(TRACKS['end_is_anchor'], bool)).sum())})" if _st == "end" else ""))

In [ ]:
# ── Part 4b — observed ALMA-C11 points, reference relations, plotting helpers ──
OBS = pd.read_csv(OBS_CSV)
OBS_S = OBS[OBS["co_det"].astype(bool) & np.isfinite(OBS["logSigmaH2"])].reset_index(drop=True)
print(f"observed: {len(OBS)} sources, {len(OBS_S)} with a surface density "
      f"({int(OBS_S['sigma_is_ll'].astype(bool).sum())} Sigma lower limits); z = {OBS['z'].min():.2f}-{OBS['z'].max():.2f}")
C_OBS, C_OBS_EDGE = "#e7298a", "#3b0f2a"
XLIM, YLIM = (-0.5, 3.6), (-4.6, 1.1)


def draw_relations(ax, xs=None, tdep_lines=(0.1, 1.0, 10.0)):
    """The RELATIONS_DRAW relation(s) with the published scatter band, RELATIONS_FAINT thin, + constant-t_dep lines. Returns legend items."""
    xs = np.linspace(XLIM[0] - 0.5, XLIM[1] + 0.5, 80) if xs is None else xs
    items = []
    for t, ls in zip(tdep_lines, (":", "-", "--")):
        ax.plot(xs, xs + 6 - np.log10(t * 1e9), color="0.65", ls=ls, lw=0.9, zorder=1)
        ax.text(XLIM[1] - 0.05, XLIM[1] - 0.05 + 6 - np.log10(t * 1e9) - 0.25, rf"$t_{{\rm dep}}$={t:g} Gyr",
                fontsize=7.5, color="0.45", ha="right", rotation=42, rotation_mode="anchor")
    for key in RELATIONS_DRAW:                  # the reference relation with its band; every relation stays in the summary offsets
        rel = kl.RELATIONS[key]
        y = rel["A"] + rel["N"] * xs
        ax.fill_between(xs, y - rel["sig"], y + rel["sig"], color=rel["color"], alpha=0.10, lw=0, zorder=0)
        ax.plot(xs, y, color=rel["color"], lw=1.4, ls=rel["ls"], zorder=2)
        items.append((Line2D([], [], color=rel["color"], lw=1.4, ls=rel["ls"]),
                      rf"{rel['label']} $\pm$ {rel['scat_label']}"))
    for key in RELATIONS_FAINT:                 # orientation only: thin, no band
        rel = kl.RELATIONS[key]
        ax.plot(xs, rel["A"] + rel["N"] * xs, color=rel["color"], lw=0.9, ls=rel["ls"], alpha=0.55, zorder=1)
        items.append((Line2D([], [], color=rel["color"], lw=0.9, ls=rel["ls"], alpha=0.55), rf"{rel['label']} (for reference)"))
    return items


def overlay_obs(ax, annotate=True, alpha=1.0):
    """ALMA-C11 points: filled = size measured (asymmetric errors), open + slope-1 arrow = unresolved
    (both Sigma lower limits). `alpha` < 1 fades them (panels outside the observed z range). Returns legend items."""
    for _, r in OBS_S.iterrows():
        x, y = float(r["logSigmaH2"]), float(r["logSigmaSFR"])
        if bool(r["sigma_is_ll"]):
            ax.annotate("", xy=(x + 0.3, y + 0.3), xytext=(x, y),
                        arrowprops=dict(arrowstyle="-|>", color=C_OBS_EDGE, lw=1.2, mutation_scale=10, alpha=alpha), zorder=8)
            ax.scatter(x, y, s=110, marker="o", facecolors="white", edgecolors=C_OBS_EDGE, linewidths=1.4, alpha=alpha, zorder=9)
            ax.scatter(x, y, s=40, marker="o", c=C_OBS, edgecolors="none", alpha=alpha, zorder=10)
        else:
            elo = r["logSigmaSFR_elo"] if np.isfinite(r["logSigmaSFR_elo"]) else 0.0
            ehi = r["logSigmaSFR_ehi"] if np.isfinite(r["logSigmaSFR_ehi"]) else 0.0
            elo = min(elo, y - YLIM[0])                     # unbounded lower errors run to the axis floor
            xe = r["logSigmaH2_err"] if np.isfinite(r["logSigmaH2_err"]) else 0.0
            ax.errorbar(x, y, xerr=xe, yerr=[[elo], [ehi]], fmt="none", ecolor=C_OBS_EDGE, elinewidth=1.0, alpha=alpha, zorder=8)
            ax.scatter(x, y, s=110, marker="o", c=C_OBS, edgecolors=C_OBS_EDGE, linewidths=1.4, alpha=alpha, zorder=9)
        if annotate:
            ax.annotate(str(r["id"]), (x, y), xytext=(5, 4), textcoords="offset points", fontsize=7, color="0.25", zorder=11)
    return [(Line2D([], [], marker="o", ls="", ms=9, mfc=C_OBS, mec=C_OBS_EDGE),
             rf"ALMA-C11 QGs, $z\approx0.4$ (N={len(OBS_S)}; size measured)"),
            (Line2D([], [], marker="o", ls="", ms=9, mfc="white", mec=C_OBS_EDGE), r"ALMA-C11 unresolved: $\Sigma$ lower limits")]


def ks_axes(ax, ap_label, sfr_note="SFR over 100 Myr (stars)"):
    conv = r"$0.5M/\pi R_{50}^2$" if ap_label.startswith("R50") else r"$M(<r)/\pi r^2$"
    he = rf", $\times{HE_FACTOR:g}$ He" if HE_FACTOR == kl.HE_FACTOR else rf", $\times{HE_FACTOR:g}$ $\neq$ He 1.36!"
    ax.set_xlabel(r"$\log(\Sigma_{\rm H_2}/M_\odot\,{\rm pc}^{-2})$  [" + AP_TITLE[ap_label] + ", " + conv + he + "]")
    ax.set_ylabel(r"$\log(\Sigma_{\rm SFR}/M_\odot\,{\rm yr}^{-1}\,{\rm kpc}^{-2})$  [" + sfr_note + "]")
    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.tick_params(direction="in", top=True, right=True); ax.grid(alpha=0.15, lw=0.6)


def _sel_ap(ap_label, pop="Q"):
    return TRACKS[(_s(TRACKS["ap_label"]) == ap_label) & (_s(TRACKS["pop"]) == pop)]


def stage_medians(T, xcol="logSigmaH2", ycol="logSigmaSFR_obs", stages=STAGES_PLOT, nmin=5):
    """Per-stage median (16-84) of x and y over rows with both finite and the SFR not censored (keys x, y, ...
    present when >= nmin such rows). Also `n_all`, `x_all`, `y_all`: the median over ALL rows with finite x, y —
    with `ycol` = the fiducial `logSigmaSFR` the censored rows enter at their one-particle floor, so `y_all` is an
    UPPER LIMIT on the stage median (present when >= nmin rows; equals the uncensored median when nothing is censored)."""
    stg, ul = _s(T["stage"]), np.asarray(T["is_ul"], bool)
    x, y = np.asarray(T[xcol], float), np.asarray(T[ycol], float)
    out = []
    for st in stages:
        m_all = (stg == st) & np.isfinite(x) & np.isfinite(y)
        m = m_all & ~ul
        d = dict(stage=st, n=int(m.sum()), n_ul=int(((stg == st) & np.isfinite(x) & ul).sum()), n_all=int(m_all.sum()))
        if m_all.sum() >= nmin:
            d.update(x_all=np.median(x[m_all]), y_all=np.median(y[m_all]))
        if m.sum() >= nmin:
            d.update(x=np.median(x[m]), x16=np.percentile(x[m], 16), x84=np.percentile(x[m], 84),
                     y=np.median(y[m]), y16=np.percentile(y[m], 16), y84=np.percentile(y[m], 84))
        out.append(d)
    return out

In [ ]:
# ── Part 4c — binned average tracks: anchor properties, bin schemes, panel / figure helpers ──
# One figure per scheme (R50(H2) only; Figure 0 below shows the anchors alone), three panels (the anchor-redshift ranges); no individual galaxies
# are drawn, only the per-stage median track of every bin over STAGES_DRAW (no bars). Bins are properties of the
# GALAXY, all evaluated at its anchor (catalogue values + the face-on member R50*), so a galaxy keeps
# its bin along the whole track. Quantile edges are computed on the full Q sample (same bins in
# every panel) unless EDGES_PER_PANEL is set.
Z_PANELS = [(0.0, 0.55, r"anchors $z\leq0.5$ (ALMA-C11 range, $z\approx0.34$–$0.43$)"),
            (0.55, 1.05, r"anchors $0.7\leq z\leq1$"),
            (1.05, 2.5, r"anchors $1.15\leq z\leq2$")]
Z_ALL = [(-1.0, 99.0, r"all anchors, $0.3\leq z\leq2$")]      # the no-redshift-separation panel (Figure 0c, the *_allz binned figures)
PLOT_APERTURES = [FIDUCIAL_AP]     # R50(H2) only: a fixed aperture changes the Sigma definition w.r.t. the relation / observed points
NMIN_BIN = 4                       # uncensored galaxies needed for a stage median inside one bin
EDGES_PER_PANEL = False            # True: terciles recomputed inside every z panel (balanced N, z-dependent bins)
SHOW_ALL_MEDIAN = True             # thin black dashed track = median of ALL Q galaxies of the panel
BIN_COLORS = ["#2166ac", "#4dac26", "#d6604d", "#762a83"]

# ── per-galaxy anchor properties (one row per Q galaxy) ──
# catalogue properties come from the anchor row; the plane quantities (own Sigma_H2, Sigma_SFR, region w.r.t. KS_REF)
# from the `end` row = the drawn endpoint (identical to the anchor unless the anchor holds no measurable H2)
_A = TRACKS[(_s(TRACKS["stage"]) == "anchor") & (_s(TRACKS["pop"]) == "Q") & (_s(TRACKS["ap_label"]) == FIDUCIAL_AP)]
_Aend = TRACKS[(_s(TRACKS["stage"]) == "end") & (_s(TRACKS["pop"]) == "Q") & (_s(TRACKS["ap_label"]) == FIDUCIAL_AP)]
_e2i = {g: i for i, g in enumerate(np.asarray(_Aend["gkey"]).astype(str))}
_ie = np.array([_e2i.get(g, -1) for g in np.asarray(_A["gkey"]).astype(str)], int)


def _end_col(name):
    """Column `name` of the galaxy's `end` row, aligned to the anchor rows (NaN when it has no end row)."""
    v = np.asarray(_Aend[name], float)
    out = np.full(len(_A), np.nan)
    out[_ie >= 0] = v[_ie[_ie >= 0]]
    return out


with np.errstate(divide="ignore", invalid="ignore"):
    _mdust, _mstar = np.asarray(_A["mdust_cat"], float), np.asarray(_A["mstar_cat"], float)
    _r50s = np.asarray(_A["r50_star"], float)                      # face-on member stellar half-mass radius [kpc]
    _r50s = np.where(np.isfinite(_r50s), _r50s, np.asarray(_A["r50star_cat"], float) / (1.0 + np.asarray(_A["z"], float)))   # catalogue radii are comoving
    _mh2 = np.asarray(_A["mh2_cat"], float)                        # catalogue H2 (hydrogen-only, like the particles)
    ANCHOR_PROPS = Table(dict(
        gkey=np.asarray(_A["gkey"]).astype(str), anchor_z=np.asarray(_A["anchor_z"], float),
        log_sfr_anchor=np.log10(np.asarray(_A["sfr_cat"], float)),          # catalogue (instantaneous) SFR
        log_mstar_anchor=np.asarray(_A["log_mstar_anchor"], float),
        log_fdust_anchor=np.log10(_mdust / _mstar),                          # specific dust mass M_dust/M_*
        log_sigdust_anchor=np.log10(0.5 * _mdust / (np.pi * _r50s ** 2) / 1e6),   # 0.5 M_dust / pi R50*^2 [Msun/pc^2]
        log_sigh2_anchor=_end_col("logSigmaH2"),                            # the plane's own x at the track end (R50_H2 row, x1.36 He; NaN if R50_H2 undefined)
        log_fh2_anchor=np.log10(HE_FACTOR * _mh2 / _mstar),                 # molecular fraction M_H2/M_* (x1.36 He: observed mu_H2 scale)
        agn_class=_s(_A["agn_class"])))
print(f"anchor properties: {len(ANCHOR_PROPS)} Q galaxies; "
      f"AGN classes: {dict(zip(*np.unique(ANCHOR_PROPS['agn_class'], return_counts=True)))}")
print("finite bin values: " + ", ".join(f"{c}={int(np.isfinite(np.asarray(ANCHOR_PROPS[c], float)).sum())}"
      for c in ("log_sfr_anchor", "log_mstar_anchor", "log_fdust_anchor", "log_sigdust_anchor", "log_sigh2_anchor", "log_fh2_anchor")))

# ── where the anchor sits w.r.t. the drawn relation: Delta = log Sigma_SFR - relation(log Sigma_H2), R50_H2 row ──
# "on" the relation = |Delta| <= KS_BAND (the published scatter of the relation). A censored anchor (SFR = 0 over
# 100 Myr, y = the one-particle floor) counts as "below" only when its floor is already below the band; if the
# floor reaches the band the region is undetermined (region "undef", not binned). NaN x (R50_H2 undefined) -> "undef".
KS_REF = RELATIONS_DRAW[0]
KS_BAND = float(kl.RELATIONS[KS_REF]["sig"])


def ks_region(x, y, ul, ref=None, band=None):
    """Region of the points (x, y) w.r.t. relation `ref` (default KS_REF) with half-width `band` (default that relation's
    published scatter): 'below' / 'within' / 'above'; 'undef' when x or
    y is NaN, or when the point is censored (`ul`, y = the one-particle floor) and its floor reaches the band.
    Returns (region str array, Delta = y - relation(x))."""
    ref = KS_REF if ref is None else ref
    band = float(kl.RELATIONS[ref]["sig"]) if band is None else float(band)
    x, y, ul = np.asarray(x, float), np.asarray(y, float), np.asarray(ul, bool)
    with np.errstate(invalid="ignore"):
        off = y - kl.relation_y(ref, x)
    fin = np.isfinite(off)
    reg = np.full(len(x), "undef", dtype=object)
    reg[fin & ~ul & (off < -band)] = "below"
    reg[fin & ~ul & (np.abs(off) <= band)] = "within"
    reg[fin & ~ul & (off > band)] = "above"
    reg[fin & ul & (off < -band)] = "below"
    return reg.astype(str), off


_ysfr, _ul = _end_col("logSigmaSFR"), _end_col("is_ul") > 0                 # at the track end (`end` row)
_reg, _off = ks_region(_end_col("logSigmaH2"), _ysfr, _ul)
ANCHOR_PROPS["log_sigsfr_anchor"] = _ysfr                 # fiducial y at the anchor (the floor where censored)
ANCHOR_PROPS["is_ul_anchor"] = _ul
ANCHOR_PROPS["ks_off_anchor"] = _off                      # Delta [dex] w.r.t. KS_REF
ANCHOR_PROPS["ks_region_anchor"] = _reg
print(f"track-end region w.r.t. {KS_REF} (band ±{KS_BAND:.2f} dex; `end` rows, {int((_ie >= 0).sum())}/{len(_A)} galaxies have one): "
      + ", ".join(f"{c}={int((_reg == c).sum())}" for c in ("below", "within", "above", "undef"))
      + f"  [censored anchors: {int(_ul.sum())}, of which below the band: {int((_ul & (_reg == 'below')).sum())}]")

BIN_SCHEMES = {
    "sfr":     dict(col="log_sfr_anchor", q=3, title="SFR at the anchor (catalogue, instantaneous)",
                    label=r"$\log({\rm SFR_{anc}}/M_\odot\,{\rm yr}^{-1})$"),
    "mstar":   dict(col="log_mstar_anchor", q=3, title="stellar mass at the anchor",
                    label=r"$\log(M_\star^{\rm anc}/M_\odot)$"),
    "fdust":   dict(col="log_fdust_anchor", q=3, title=r"dust fraction $M_{\rm dust}/M_\star$ at the anchor",
                    label=r"$\log(M_{\rm dust}/M_\star)_{\rm anc}$"),
    "sigdust": dict(col="log_sigdust_anchor", q=3, title=r"dust surface density $0.5M_{\rm dust}/\pi R_{50\star}^2$ at the anchor",
                    label=r"$\log(\Sigma_{\rm dust}^{\rm anc}/M_\odot\,{\rm pc}^{-2})$"),
    "sigh2":   dict(col="log_sigh2_anchor", q=3, title=r"own $\Sigma_{\rm H_2}$ at the anchor ($R_{50}({\rm H_2})$ row, $\times1.36$ He; the plane's $x$)",
                    label=r"$\log(\Sigma_{\rm H_2}^{\rm anc}/M_\odot\,{\rm pc}^{-2})$"),
    "fh2":     dict(col="log_fh2_anchor", q=3, title=r"molecular fraction $M_{\rm H_2}/M_\star$ at the anchor ($\times1.36$ He)",
                    label=r"$\log(M_{\rm H_2}/M_\star)_{\rm anc}$"),
    "ksreg":   dict(col="ks_region_anchor", categories=["below", "within", "above"],   # region at the track end (`end` row)
                    cat_labels=[rf"below {KS_REF} ($\Delta < -{KS_BAND:.2f}$ dex)", rf"on {KS_REF} ($|\Delta| \leq {KS_BAND:.2f}$ dex)",
                                rf"above {KS_REF} ($\Delta > {KS_BAND:.2f}$ dex)"],
                    title=rf"where the track ends w.r.t. {KS_REF}: $\Delta = \log\Sigma_{{\rm SFR}} - \log\Sigma_{{\rm SFR}}^{{\rm {KS_REF}}}(\Sigma_{{\rm H_2}})$",
                    label=r"$\Delta$"),
    "agn":     dict(col="agn_class", categories=["weak", "intermediate", "strong"],
                    title="AGN coupling class during the quenching phase",
                    label="coupling"),
}

# region colours / labels shared by Figure 0, the ksreg scheme and Figure 3
REGION_ORDER = ["below", "within", "above", "undef"]
REGION_COLOR = dict(zip(BIN_SCHEMES["ksreg"]["categories"], BIN_SCHEMES["ksreg"].get("colors", BIN_COLORS)), undef="0.35")
REGION_LABEL = dict(zip(BIN_SCHEMES["ksreg"]["categories"], BIN_SCHEMES["ksreg"]["cat_labels"]),
                    undef="censored, floor reaches the band")


def assign_bins(props, scheme, mask=None):
    """Bin index (-1 = unbinned) per row of `props` + bin descriptors [(label, colour), ...].

    Quantile schemes (`q` terciles by default) use the rows in `mask` (all rows when None) to set the
    edges; categorical schemes take `categories` in order (anything else -> -1), with optional display
    names `cat_labels` and per-bin `colors` (default BIN_COLORS)."""
    v = props[scheme["col"]]
    idx = np.full(len(props), -1, int)
    colors = list(scheme.get("colors", BIN_COLORS))
    if "categories" in scheme:
        cats = list(scheme["categories"])
        labs = list(scheme.get("cat_labels", cats))
        v = np.asarray(v).astype(str)
        for k, c in enumerate(cats):
            idx[v == c] = k
        return idx, [(labs[k], colors[k]) for k in range(len(cats))]
    v = np.asarray(v, float)
    use = np.isfinite(v) & (np.ones(len(v), bool) if mask is None else np.asarray(mask, bool))
    q = int(scheme.get("q", 3))
    edges = np.percentile(v[use], np.linspace(0, 100, q + 1)[1:-1]) if use.sum() >= q else np.array([])
    idx[np.isfinite(v)] = np.searchsorted(edges, v[np.isfinite(v)], side="right")
    bounds = np.r_[-np.inf, edges, np.inf]
    descr = []
    for k in range(len(bounds) - 1):
        lo, hi = bounds[k], bounds[k + 1]
        if not np.isfinite(lo):
            lab = rf"{scheme['label']} $\leq$ {hi:.2f}"
        elif not np.isfinite(hi):
            lab = rf"{scheme['label']} $>$ {lo:.2f}"
        else:
            lab = rf"{lo:.2f} $<$ {scheme['label']} $\leq$ {hi:.2f}"
        descr.append((lab, colors[k]))
    return idx, descr


def draw_median_track(ax, T, color, ycol="logSigmaSFR", lw=2.2, ls="-", zorder=6, ms=1.0):
    """Per-stage medians of T (stage_medians over STAGES_DRAW) joined in stage order; markers = stage symbols. No bars.
    A stage with < NMIN_BIN uncensored galaxies but >= NMIN_BIN in total (typically the anchor of a bin full of
    SFR = 0 galaxies) is drawn at its all-rows median with the censored SFRs at their floor: open marker + down
    arrow = an upper limit on the stage median."""
    meds = stage_medians(T, ycol=ycol, stages=STAGES_DRAW, nmin=NMIN_BIN)
    pts = [(m["x"], m["y"], m["stage"], False) if "x" in m else (m["x_all"], m["y_all"], m["stage"], True)
           for m in meds if ("x" in m) or ("x_all" in m)]
    if len(pts) >= 2:
        ax.plot([p[0] for p in pts], [p[1] for p in pts], ls=ls, color=color, lw=lw, alpha=0.9, zorder=zorder)
    for x, y, st, is_ul in pts:
        s = (110 if st in END_STAGES else 70) * ms
        if is_ul:
            ax.scatter(x, y, s=s, marker=STAGE_MARKER[st], facecolors="white", edgecolors=color, linewidths=1.4, zorder=zorder + 1)
            ax.annotate("", xy=(x, y - 0.35), xytext=(x, y), zorder=zorder + 1,
                        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.2, mutation_scale=9, shrinkA=4, shrinkB=0))
        else:
            ax.scatter(x, y, s=s, marker=STAGE_MARKER[st], c=[color], edgecolors="k" if st in END_STAGES else "none",
                       linewidths=0.9, zorder=zorder + 1)
    return meds


def binned_panel(ax, ap_label, zlo, zhi, scheme, ycol="logSigmaSFR", show_sf=True, obs_alpha=1.0, title=None):
    """One KS panel: median tracks of every bin of `scheme` for the Q galaxies whose anchor lies in
    (zlo, zhi]; SF-control cloud of the same anchors; relations; observed points. Returns
    [(bin label, colour, n_gal, meds), ...] (n_gal = galaxies of the panel in the bin)."""
    T = _sel_ap(ap_label, "Q")
    az = np.asarray(T["anchor_z"], float)
    T = T[(az > zlo) & (az <= zhi)]
    gk = np.asarray(T["gkey"]).astype(str)
    in_panel = (ANCHOR_PROPS["anchor_z"] > zlo) & (ANCHOR_PROPS["anchor_z"] <= zhi)
    idx, descr = assign_bins(ANCHOR_PROPS, scheme, mask=in_panel if EDGES_PER_PANEL else None)
    g2b = dict(zip(np.asarray(ANCHOR_PROPS["gkey"]).astype(str), idx))
    b = np.array([g2b.get(g, -1) for g in gk], int)
    if show_sf:
        S = _sel_ap(ap_label, "SF")
        azs = np.asarray(S["anchor_z"], float)
        S = S[(azs > zlo) & (azs <= zhi)]
        ax.scatter(np.asarray(S["logSigmaH2"], float), np.asarray(S[ycol], float), s=7, c="0.6", alpha=0.3, lw=0, zorder=2)
    if SHOW_ALL_MEDIAN:
        draw_median_track(ax, T, "k", ycol=ycol, lw=1.2, ls="--", zorder=5, ms=0.45)
    out = []
    for k, (lab, col) in enumerate(descr):
        Tb = T[b == k]
        n_gal = len(np.unique(np.asarray(Tb["gkey"]).astype(str)))
        meds = draw_median_track(ax, Tb, col, ycol=ycol) if n_gal else []
        out.append((lab, col, n_gal, meds))
    rel_items = draw_relations(ax)
    obs_items = overlay_obs(ax, annotate=False, alpha=obs_alpha)
    ks_axes(ax, ap_label, "SFR over 100 Myr (stars)" if "inst" not in ycol else "instantaneous gas SFR")
    if title:
        ax.set_title(title, loc="left", fontsize=11)
    # bin legend (N differs per panel) in the empty corner below the t_dep = 10 Gyr line
    ax.legend([Line2D([], [], color=c, lw=2.2, marker="o", mec="k", ms=7) for _, c, _, _ in out]
              + ([Line2D([], [], color="k", lw=1.2, ls="--")] if SHOW_ALL_MEDIAN else []),
              [f"{l}  (N={n})" for l, _, n, _ in out] + (["all Q of this panel"] if SHOW_ALL_MEDIAN else []),
              loc="lower right", frameon=False, fontsize=8.5, title=scheme["title"], title_fontsize=9)
    return out, rel_items, obs_items


def binned_figure(scheme_key, ap_label, ycol="logSigmaSFR", save=True, zpanels=Z_PANELS):
    """One figure per (scheme, aperture): one panel per entry of `zpanels` (the three anchor-z ranges, or Z_ALL for a single
    all-redshift panel -> file suffix _allz); legends for stages / relations / observed. Bin edges are the same in every case."""
    scheme = BIN_SCHEMES[scheme_key]
    nz = len(zpanels)
    fig, axs = plt.subplots(1, nz, figsize=(6.3 * nz + 0.1, 6.2 if nz > 1 else 7.2), squeeze=False)
    res = []
    for j, (ax, (zlo, zhi, ttl)) in enumerate(zip(axs[0], zpanels)):
        out, rel_items, obs_items = binned_panel(ax, ap_label, zlo, zhi, scheme, ycol=ycol,
                                                 obs_alpha=1.0 if (j == 0 or nz == 1) else 0.3,
                                                 title=(f"({'abc'[j]}) " if nz > 1 else "") + ttl)
        res.append(((zlo, zhi), out))
    # shared legend row under the panels: stage symbols, then relations + observed points
    _stage_items = [(Line2D([], [], marker=STAGE_MARKER[st], ls="", ms=8, mfc="0.4", mec="k" if st in END_STAGES else "none"),
                     STAGE_LABEL[st]) for st in STAGES_DRAW]
    _misc = [(Line2D([], [], color="0.4", lw=2.2), "bin median track (stages in time order)"),
             (Line2D([], [], marker="o", ls="", ms=8, mfc="white", mec="0.4", mew=1.4),
              rf"open + $\downarrow$: < {NMIN_BIN} uncensored galaxies, median with SFR = 0 at the floor (upper limit)"),
             (Line2D([], [], marker="o", ls="", ms=5, mfc="0.6", mec="none"), "SF controls at their anchor")]
    _items = _stage_items + _misc + rel_items + obs_items
    fig.legend([h for h, _ in _items], [l for _, l in _items], loc="lower center", ncol=5 if nz > 1 else 2, frameon=False,
               fontsize=8.5, bbox_to_anchor=(0.5, -0.005 if nz > 1 else -0.01))
    fig.suptitle(f"SIMBA-25 quenched galaxies — average KS tracks binned by {scheme['title']}" + ("" if nz > 1 else ", all anchors together") + "   "
                 f"[{AP_TITLE[ap_label]}; " + ("100 Myr archaeological SFR" if "inst" not in ycol else "instantaneous SFR") + "]"
                 + (f"\n{END_DEF}" if "end" in STAGES_DRAW else ""), y=0.995, fontsize=12 if nz > 1 else 10)
    plt.tight_layout(rect=(0, 0.07 if nz > 1 else 0.11, 1, 0.97 if nz > 1 else 0.94))
    if save:
        fig.savefig(os.path.join(FIGDIR, f"ks_tracks_binned_{scheme_key}" + ("" if nz > 1 else "_allz") + f"_{ap_label}"
                                 + ("_inst" if "inst" in ycol else "") + ".png"), dpi=160, bbox_inches="tight")
    return fig, res

In [ ]:
# ── Figure 0 — the SIMBA galaxies alone on the KS plane against the reference relation KS_REF: at the anchor and at every critical point ──
# No tracks, no observed points. Every galaxy is coloured by its region w.r.t. the relation AT THAT POINT (below /
# on / above the ±KS_BAND band; censored SFR = 0 as open downward triangles at the one-particle floor, counted as
# "below" only when the floor is already below the band). The SF controls (grey) exist at the anchor only.
#   0a  ks_anchors_<ap>.png      anchors, the three anchor-redshift panels
#   0b  ks_stages_<ap>.png       rows = STAGES_DRAW (time order) x the three anchor-redshift panels
#   0c  ks_stages_allz_<ap>.png  one panel per stage, all anchor redshifts together
# Counts per (stage, z range) -> ks_stage_regions.csv: is there a point where enough galaxies sit BELOW the relation?


def stage_panel(ax, ap_label, stage, zlo, zhi, title=None, show_sf=True):
    """Q galaxies at `stage` (rows of aperture `ap_label`, anchor z in (zlo, zhi]) coloured by their region w.r.t.
    KS_REF at that point; optional SF controls of the same anchors; the reference relation.
    Returns ({region: (n_det, n_ul)}, relation legend items)."""
    T = _sel_ap(ap_label, "Q")
    az = np.asarray(T["anchor_z"], float)
    T = T[(_s(T["stage"]) == stage) & (az > zlo) & (az <= zhi)]
    x, y, ul = np.asarray(T["logSigmaH2"], float), np.asarray(T["logSigmaSFR"], float), np.asarray(T["is_ul"], bool)
    reg, _ = ks_region(x, y, ul)
    if show_sf:
        S = _sel_ap(ap_label, "SF")
        S = S[(_s(S["stage"]) == "anchor") & (np.asarray(S["anchor_z"], float) > zlo) & (np.asarray(S["anchor_z"], float) <= zhi)]
        ax.scatter(np.asarray(S["logSigmaH2"], float), np.asarray(S["logSigmaSFR"], float), s=9, c="0.6", alpha=0.35, lw=0, zorder=2)
    counts, handles, labels = {}, [], []
    for c in REGION_ORDER:
        m = (reg == c) & np.isfinite(x) & np.isfinite(y)
        n_det, n_ul = int((m & ~ul).sum()), int((m & ul).sum())
        counts[c] = (n_det, n_ul)
        if not (n_det + n_ul):
            continue
        col = REGION_COLOR[c]
        ax.scatter(x[m & ~ul], y[m & ~ul], s=28, c=[col], edgecolors="k", linewidths=0.4, alpha=0.9, zorder=6)
        ax.scatter(x[m & ul], y[m & ul], s=40, marker="v", facecolors="none", edgecolors=col, linewidths=1.1, zorder=6)
        handles.append(Line2D([], [], marker="o" if n_det else "v", ls="", ms=6, mfc=col if n_det else "none", mec="k" if n_det else col))
        labels.append(f"{REGION_LABEL[c]}  (N={n_det}" + (f" + {n_ul} UL" if n_ul else "") + ")")
    rel_items = draw_relations(ax)
    ks_axes(ax, ap_label)
    if title:
        ax.set_title(title, loc="left", fontsize=11)
    ax.legend(handles, labels, loc="lower right", frameon=False, fontsize=8.5, title_fontsize=9,
              title=f"Q at {STAGE_LABEL[stage]} — region vs {KS_REF} (±{KS_BAND:.2f} dex) at this point")
    return counts, rel_items


def stage_figure(stages, fname, zpanels=Z_PANELS, ap_label=FIDUCIAL_AP):
    """Grid: rows = `stages` (time order) x columns = `zpanels`; with a single z range the stages go in one row.
    SF controls in the anchor panels only. Returns one summary row per (stage, z range)."""
    one_row = len(zpanels) == 1
    nr, nc = (1, len(stages)) if one_row else (len(stages), len(zpanels))
    fig, axs = plt.subplots(nr, nc, figsize=(6.3 * nc + 0.1, 6.2 * nr), squeeze=False)
    rows, k = [], 0
    for i, st in enumerate(stages):
        for j, (zlo, zhi, ttl) in enumerate(zpanels):
            ax = axs[0, i] if one_row else axs[i, j]
            counts, rel_items = stage_panel(ax, ap_label, st, zlo, zhi, show_sf=(st in END_STAGES),
                                            title=f"({'abcdefghijklmnop'[k]}) {STAGE_LABEL[st]} — {ttl}")
            k += 1
            n_tot = sum(d + u for d, u in counts.values())
            rows.append(dict(stage=st, z_lo=zlo, z_hi=zhi, n=n_tot,
                             **{f"n_{c}": d + u for c, (d, u) in counts.items()}, **{f"n_{c}_ul": u for c, (d, u) in counts.items()},
                             f_below=(counts["below"][0] + counts["below"][1]) / n_tot if n_tot else np.nan))
    _items = [(Line2D([], [], marker="o", ls="", ms=6, mfc="0.4", mec="k"), "quenched galaxy at the stage (colour = region at that point)"),
              (Line2D([], [], marker="v", ls="", ms=6, mfc="none", mec="0.4"), "censored: SFR = 0 over 100 Myr, one-particle floor"),
              (Line2D([], [], marker="o", ls="", ms=5, mfc="0.6", mec="none"), "SF controls at their anchor (endpoint panels only)")] + rel_items
    fig.legend([h for h, _ in _items], [l for _, l in _items], loc="lower center", ncol=4, frameon=False, fontsize=8.5,
               bbox_to_anchor=(0.5, -0.005 / nr))
    what = "anchors" if stages == ["anchor"] else "critical points " + " → ".join(STAGE_LABEL[s] for s in stages)
    fig.suptitle(f"SIMBA-25 quenched galaxies on the KS plane at the {what} — no tracks, no observed points   "
                 f"[{AP_TITLE[ap_label]}; 100 Myr archaeological SFR]" + (f"\n{END_DEF}" if "end" in stages else ""),
                 y=1.0 - 0.005 / nr)
    plt.tight_layout(rect=(0, 0.06 / nr, 1, 1 - 0.03 / nr))
    fig.savefig(os.path.join(FIGDIR, fname), dpi=160, bbox_inches="tight")
    plt.show()
    for r in rows:
        print(f"   {STAGE_LABEL[r['stage']]:20s} z({r['z_lo']:.2f},{r['z_hi']:.2f}]  N={r['n']:3d}  "
              + "  ".join(f"{c}: {r[f'n_{c}'] - r[f'n_{c}_ul']}" + (f"+{r[f'n_{c}_ul']}UL" if r[f'n_{c}_ul'] else "")
                          + (f" ({100 * r[f'n_{c}'] / r['n']:.0f}%)" if r["n"] else "") for c in REGION_ORDER))
    return rows


print("Figure 0a — anchors, by anchor redshift")
stage_figure(["anchor"], f"ks_anchors_{FIDUCIAL_AP}.png")
_P = ANCHOR_PROPS[np.isfinite(ANCHOR_PROPS["ks_off_anchor"]) & ~ANCHOR_PROPS["is_ul_anchor"]]
print(f"uncensored Q track ends: median Delta = {np.median(_P['ks_off_anchor']):+.2f} dex "
      f"(16-84: {np.percentile(_P['ks_off_anchor'], 16):+.2f}, {np.percentile(_P['ks_off_anchor'], 84):+.2f}) w.r.t. {KS_REF}")
print("Figure 0b — every critical point of the quench phase, by anchor redshift (rows in time order)")
_rows = stage_figure(list(STAGES_DRAW), f"ks_stages_{FIDUCIAL_AP}.png")
print("Figure 0c — every critical point, all anchor redshifts together")
_rows += stage_figure(list(STAGES_DRAW), f"ks_stages_allz_{FIDUCIAL_AP}.png", zpanels=Z_ALL)
STAGE_REGIONS = Table(rows=_rows)
STAGE_REGIONS.to_pandas().to_csv(os.path.join(KSDIR, "ks_stage_regions.csv"), index=False)
print(f"region counts per (stage, z range) -> {os.path.join(KSDIR, 'ks_stage_regions.csv')}")
# the observed side for reference (Sigma lower limits taken at face value; their slope-1 arrows only move them further below)
_oreg, _ = ks_region(OBS_S["logSigmaH2"], OBS_S["logSigmaSFR"], np.zeros(len(OBS_S), bool))
print(f"observed ALMA-C11 (N={len(OBS_S)}) w.r.t. {KS_REF}: " + ", ".join(f"{c}={int((_oreg == c).sum())}" for c in REGION_ORDER[:3]))


In [ ]:
# ── Figures 1-8 — average KS tracks in the fiducial aperture R50(H2), TWO figures per binning scheme: the three anchor-z panels
# (ks_tracks_binned_<scheme>_R50_H2.png) and all anchors together in one panel (ks_tracks_binned_<scheme>_allz_R50_H2.png; same bin edges) ──
# sfr / mstar / fdust / sigdust / own Sigma_H2 / molecular fraction at the anchor, the track end's region w.r.t. the reference relation KS_REF (B08)
# (below / on / above the band at the track end: where do the galaxies that END below, on, or above the relation come from?), then the
# AGN coupling class during the quenching phase (galaxies without a detected quench event, agn_class = no_event, are
# not binned there).
SCHEMES_PLOT = ("sfr", "mstar", "fdust", "sigdust", "sigh2", "fh2", "ksreg", "agn")
_first, _last = STAGES_DRAW[0], STAGES_DRAW[-1]
_rows = []
for _sk in SCHEMES_PLOT:
    for _ap in PLOT_APERTURES:
        res = []
        for _zp in (Z_PANELS, Z_ALL):                       # the three anchor-z panels, then all anchors in one panel
            fig, _r = binned_figure(_sk, _ap, zpanels=_zp)
            res += _r
        print(f"[{_sk} | {_ap}]")
        for (zlo, zhi), out in res:
            for lab, _, n_gal, meds in out:
                def _pt(st):     # (x, y, flag) of the drawn point: uncensored median, else the floor-included upper limit
                    m = next((m for m in meds if m["stage"] == st), None)
                    if m is None: return None
                    if "x" in m: return (m["x"], m["y"], "")
                    if "x_all" in m: return (m["x_all"], m["y_all"], "<")
                    return None
                anc, fst = _pt(_last), _pt(_first)
                n_anc = next((m for m in meds if m["stage"] == _last), None)
                print(f"   z({zlo:.2f},{zhi:.2f}]  {lab:48s} N={n_gal:3d}  "
                      + (f"{_first} ({fst[0]:+.2f},{fst[2]}{fst[1]:+.2f}) " if fst else f"{_first} n/a ")
                      + (f"{_last} ({anc[0]:+.2f},{anc[2]}{anc[1]:+.2f}) N={n_anc['n']} UL={n_anc['n_ul']}" if anc else f"{_last} n/a"))
                for m in meds:
                    _rows.append(dict(scheme=_sk, aperture=_ap, z_lo=zlo, z_hi=zhi, bin=lab, n_gal=n_gal, stage=m["stage"],
                                      n=m["n"], n_ul=m["n_ul"], n_all=m["n_all"],
                                      **{k: m.get(k, np.nan) for k in ("x", "x16", "x84", "y", "y16", "y84", "x_all", "y_all")}))
        plt.show()
BINNED_SUMMARY = Table(rows=_rows)
write_table(BINNED_SUMMARY, os.path.join(KSDIR, "ks_binned_tracks.fits"))
BINNED_SUMMARY.to_pandas().to_csv(os.path.join(KSDIR, "ks_binned_tracks.csv"), index=False)
print(f"binned-track medians: {len(BINNED_SUMMARY)} rows -> {os.path.join(KSDIR, 'ks_binned_tracks.csv')}")

# figures of apertures no longer drawn (earlier 3-aperture runs) would otherwise linger next to the current ones
_stale = [p for p in glob.glob(os.path.join(FIGDIR, "ks_tracks_binned_*.png"))
          if not any(os.path.basename(p).endswith(f"_{ap}.png") or os.path.basename(p).endswith(f"_{ap}_inst.png") for ap in PLOT_APERTURES)]
for p in _stale:
    os.remove(p)
print(f"removed {len(_stale)} stale figure(s) of apertures not in PLOT_APERTURES" + (": " + ", ".join(map(os.path.basename, _stale)) if _stale else ""))


In [ ]:
# ── Figure 2 — depletion clock: t_dep(H2) vs time since QT, by AGN coupling class ──
T = _sel_ap(FIDUCIAL_AP, "Q")
_dt = np.asarray(T["dt_from_qt_gyr"], float)
_td = np.asarray(T["tdep_gyr"], float)
_stg, _cls = _s(T["stage"]), _s(T["agn_class"])
_ok = np.isfinite(_dt) & np.isfinite(_td) & (_td > 0)
CLASSES = [("weak", "#1b9e77"), ("intermediate", "#7570b3"), ("strong", "#d95f02")]
_obs_td = OBS["tdep_Gyr"][OBS["co_det"].astype(bool)].values
_obs_td = _obs_td[np.isfinite(_obs_td)]

fig, axs = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw=dict(width_ratios=[3, 1.2]))
ax = axs[0]
for st in STAGES_PLOT:
    m = _ok & (_stg == st)
    ax.scatter(_dt[m], np.log10(_td[m]), s=16, marker=STAGE_MARKER[st], c=[STAGE_COLOR[st]], alpha=0.55,
               label=f"{STAGE_LABEL[st]} (N={int(m.sum())})", zorder=3)
_bins = np.array([-4, -2, -1, -0.5, -0.25, 0, 0.25, 0.75, 1.25, 2, 3, 5, 8])
for cname, ccol in CLASSES:
    m = _ok & (_cls == cname)
    xs_, ys_ = [], []
    for lo, hi in zip(_bins[:-1], _bins[1:]):
        mb = m & (_dt >= lo) & (_dt < hi)
        if mb.sum() >= 4:
            xs_.append(np.median(_dt[mb])); ys_.append(np.median(np.log10(_td[mb])))
    if xs_:
        ax.plot(xs_, ys_, "-", color=ccol, lw=2.2, zorder=5, label=f"median, {cname} coupling (N={int(m.sum())})")
ax.axvline(0, color="0.5", lw=0.8)
if _obs_td.size:
    ax.axhspan(np.log10(np.percentile(_obs_td, 16)), np.log10(np.percentile(_obs_td, 84)), color=C_OBS, alpha=0.15, lw=0,
               label=rf"ALMA-C11 $t_{{\rm dep}}$ 16–84 % (N={len(_obs_td)})")
ax.axhline(np.log10(kl.tdep_ms_gyr(Z_OBS)), color="0.3", ls="--", lw=1.0, label=f"Tacconi+18 MS at z={Z_OBS}")
ax.set_xlabel(r"$t - t_{\rm QT}$ [Gyr]"); ax.set_ylabel(r"$\log(t_{\rm dep}/{\rm Gyr})$  [" + AP_TITLE[FIDUCIAL_AP] + ", 100 Myr SFR]")
ax.set_title("(a) molecular depletion time along the quench clock", loc="left"); ax.legend(ncol=2, fontsize=8)
ax = axs[1]
_anc = _ok & (_stg == "anchor")
ax.hist(np.log10(_td[_anc]), bins=np.linspace(-1.5, 2.5, 25), color="0.6", label=f"SIMBA anchors (N={int(_anc.sum())})")
if _obs_td.size:
    ax.hist(np.log10(_obs_td), bins=np.linspace(-1.5, 2.5, 25), histtype="step", lw=2, color=C_OBS, label="ALMA-C11")
ax.set_xlabel(r"$\log(t_{\rm dep}/{\rm Gyr})$"); ax.set_ylabel("N"); ax.legend(fontsize=8)
ax.set_title("(b) at the anchor vs observed", loc="left")
plt.tight_layout(); fig.savefig(os.path.join(FIGDIR, "ks_tracks_tdep_clock.png"), dpi=160); plt.show()

### Figure 3 — is the offset from the relation a Σ-definition effect? A fair comparison

With one radius for both surface densities the aperture cancels out of the offset from Kennicutt 98:

$$\Delta_{\rm K98} = \log\Sigma_{\rm SFR} - {\rm K98}(\Sigma_{\rm H_2}) = (6 - A_{\rm K98}) - \log t_{\rm dep}[{\rm yr}] - 0.4\,\log\Sigma_{\rm H_2},$$

i.e. it is the H$_2$ depletion time up to a weak size term (more extended → *higher* Δ). SIMBA forms stars as
SFR = ε$_{\rm ff}$ ρ$_{\rm H_2}$/t$_{\rm ff}$ with ε$_{\rm ff}$ = 0.02 on the same subgrid f$_{\rm H_2}$ we measure, so
its instantaneous t$_{\rm dep}$ = t$_{\rm ff}$/ε$_{\rm ff}$ ≈ 0.7–6 Gyr for n$_{\rm H}$ = 10–0.13 cm$^{-3}$: the simulation *is*
a KS relation by construction. Panel (a) shows Δ against t$_{\rm dep}$ with the iso-Σ$_{\rm H_2}$ lines of that identity,
(b) the t$_{\rm dep}$ distributions (100 Myr and instantaneous SFR; SF controls; observed) against the SF-law band and the
Tacconi+18 main sequence. The two "fair" references: (c) the same Σ$_{\rm H_2}$ against an **H$_2$-based** relation
(Bigiel+08, ±0.20 dex) instead of the total-gas K98; (d) **total neutral gas** Σ(HI+H$_2$)×1.36 inside R$_{50}$(H$_2$)
against K98 / RK19, which is what those relations were calibrated on — the observed points have no HI and become
lower limits on Σ$_{\rm gas}$. Counts → `ks_fair_comparison.csv`.


In [ ]:
# ── Figure 3 — Delta vs t_dep + fair references (H2 relation; total neutral gas) at the track end ──
FAIR_STAGE = STAGES_DRAW[-1]
_TQ = _sel_ap(FIDUCIAL_AP, "Q"); _TQ = _TQ[_s(_TQ["stage"]) == FAIR_STAGE]
_TS = _sel_ap(FIDUCIAL_AP, "SF")
_col = lambda T, c: np.asarray(T[c], float)
xq, yq, ulq = _col(_TQ, "logSigmaH2"), _col(_TQ, "logSigmaSFR"), np.asarray(_TQ["is_ul"], bool)
xs_, ys_, uls = _col(_TS, "logSigmaH2"), _col(_TS, "logSigmaSFR"), np.asarray(_TS["is_ul"], bool)
gq, gs = _col(_TQ, "logSigmaGas"), _col(_TS, "logSigmaGas")
A_REF, N_REF = float(kl.RELATIONS[KS_REF]["A"]), float(kl.RELATIONS[KS_REF]["N"])   # Delta_ref = (6 - A) - log t_dep - (N - 1) log Sigma_H2
C_REF = kl.RELATIONS[KS_REF]["color"]
EPS_FF, NH_SF = 0.02, (0.13, 10.0)                       # SIMBA: SFR = eps_ff rho_H2 / t_ff; n_H range of its star-forming gas


def _tdep_sflaw_gyr(n_h):
    """t_ff / eps_ff [Gyr] at hydrogen number density n_H [cm^-3] (rho = n_H m_p / X_H, X_H = 0.76)."""
    rho = np.asarray(n_h, float) * 1.6726e-24 / 0.76
    return np.sqrt(3 * np.pi / (32 * 6.674e-8 * rho)) / 3.156e7 / EPS_FF / 1e9


def _rel_line(ax, key, band=True, lw=1.4, alpha=1.0, zorder=2):
    rel = kl.RELATIONS[key]
    xs = np.linspace(XLIM[0] - 0.5, XLIM[1] + 0.5, 80)
    y = rel["A"] + rel["N"] * xs
    if band:
        ax.fill_between(xs, y - rel["sig"], y + rel["sig"], color=rel["color"], alpha=0.10, lw=0, zorder=0)
    ax.plot(xs, y, color=rel["color"], lw=lw, ls=rel["ls"], alpha=alpha, zorder=zorder)
    return (Line2D([], [], color=rel["color"], lw=lw, ls=rel["ls"], alpha=alpha),
            rf"{rel['label']}" + (rf" $\pm$ {rel['scat_label']}" if band else ""))


def _region_scatter(ax, x, y, ul, ref, show_sf=None):
    """Q points coloured by region w.r.t. `ref` (its own scatter as the band); returns (counts, legend items)."""
    reg, _ = ks_region(x, y, ul, ref=ref)
    if show_sf is not None:
        ax.scatter(show_sf[0], show_sf[1], s=9, c="0.6", alpha=0.35, lw=0, zorder=2)
    counts, items = {}, []
    for c in REGION_ORDER:
        m = (reg == c) & np.isfinite(x) & np.isfinite(y)
        n_det, n_ul = int((m & ~ul).sum()), int((m & ul).sum())
        counts[c] = (n_det, n_ul)
        if not (n_det + n_ul):
            continue
        col = REGION_COLOR[c]
        ax.scatter(x[m & ~ul], y[m & ~ul], s=28, c=[col], edgecolors="k", linewidths=0.4, alpha=0.9, zorder=6)
        ax.scatter(x[m & ul], y[m & ul], s=40, marker="v", facecolors="none", edgecolors=col, linewidths=1.1, zorder=6)
        lab = {"below": "below", "within": "on", "above": "above", "undef": "censored, floor reaches the band"}[c]
        items.append((Line2D([], [], marker="o" if n_det else "v", ls="", ms=6, mfc=col if n_det else "none", mec="k" if n_det else col),
                      f"{lab} {ref if c != 'undef' else ''}  (N={n_det}" + (f" + {n_ul} UL" if n_ul else "") + ")"))
    return counts, items


def _count_row(panel, xname, ref, popname, counts):
    n = sum(d + u for d, u in counts.values())
    return dict(panel=panel, x=xname, relation=ref, band_dex=float(kl.RELATIONS[ref]["sig"]), pop=popname, stage=FAIR_STAGE if popname == "Q" else "anchor",
                n=n, **{f"n_{c}": d + u for c, (d, u) in counts.items()}, **{f"n_{c}_ul": u for c, (d, u) in counts.items()},
                f_below=(counts["below"][0] + counts["below"][1]) / n if n else np.nan)


FAIR_ROWS = []
fig, axs = plt.subplots(2, 2, figsize=(14.5, 12.4))

# (a) Delta_K98 vs t_dep: the identity, coloured by log Sigma_H2 (the only other term)
ax = axs[0, 0]
ltq = np.log10(_col(_TQ, "tdep_gyr") * 1e9)                     # NaN where censored
dq = yq - kl.relation_y(KS_REF, xq)
lts = np.log10(_col(_TS, "tdep_gyr") * 1e9)
ds = ys_ - kl.relation_y(KS_REF, xs_)
ok_q, ok_s = np.isfinite(ltq) & np.isfinite(dq), np.isfinite(lts) & np.isfinite(ds)
cmap, norm = plt.get_cmap("viridis"), plt.Normalize(0.5, 3.5)
tt = np.linspace(8.0, 10.6, 60)
for xv in (0.5, 1.5, 2.5, 3.5):
    ax.plot(tt, (6 - A_REF) - tt - (N_REF - 1) * xv, color=cmap(norm(xv)), lw=1.0, ls="--", zorder=1)
    ax.text(8.25, (6 - A_REF) - 8.25 - (N_REF - 1) * xv + 0.04, rf"$\log\Sigma_{{\rm H_2}}={xv}$", color=cmap(norm(xv)), fontsize=7.5, ha="left", rotation=-38, rotation_mode="anchor")
ax.axhspan(-KS_BAND, KS_BAND, color=C_REF, alpha=0.10, lw=0, zorder=0)
ax.axhline(0, color=C_REF, lw=1.2, zorder=1)
ax.scatter(lts[ok_s], ds[ok_s], s=9, c="0.6", alpha=0.4, lw=0, zorder=2, label=f"SF controls at their anchor (N={int(ok_s.sum())})")
sc = ax.scatter(ltq[ok_q], dq[ok_q], c=xq[ok_q], cmap=cmap, norm=norm, s=26, edgecolors="k", linewidths=0.35, zorder=5,
                label=f"quenched at the {STAGE_LABEL[FAIR_STAGE]} (N={int(ok_q.sum())}; {int((np.isfinite(dq) & ulq).sum())} censored not shown)")
_lo = np.log10(OBS_S["tdep_Gyr"].values * 1e9); _do = OBS_S["logSigmaSFR"].values - kl.relation_y(KS_REF, OBS_S["logSigmaH2"].values)
ax.scatter(_lo, _do, s=90, c=C_OBS, edgecolors=C_OBS_EDGE, linewidths=1.2, zorder=7, label=f"ALMA-C11 QGs (N={len(OBS_S)})")
ax.axvline(np.log10(kl.tdep_ms_gyr(Z_OBS) * 1e9), color="0.3", ls=":", lw=1.0, label=f"Tacconi+18 MS at z={Z_OBS}")
plt.colorbar(sc, ax=ax, pad=0.01, label=r"$\log(\Sigma_{\rm H_2}/M_\odot\,{\rm pc}^{-2})$")
ax.set_xlim(8.2, 10.6); ax.set_ylim(-1.8, 1.8)
ax.set_xlabel(r"$\log(t_{\rm dep}/{\rm yr})$  [$\Sigma_{\rm H_2}\times10^6/\Sigma_{\rm SFR}$, " + AP_TITLE[FIDUCIAL_AP] + "]")
ax.set_ylabel(rf"$\Delta_{{\rm {KS_REF}}} = \log\Sigma_{{\rm SFR}} - {{\rm {KS_REF}}}(\Sigma_{{\rm H_2}})$ [dex]")
ax.set_title(rf"(a) $\Delta_{{\rm {KS_REF}}} = {6 - A_REF:.2f} - \log t_{{\rm dep}} {'-' if N_REF >= 1 else '+'} {abs(N_REF - 1):.2f}\log\Sigma_{{\rm H_2}}$: the offset is the depletion time", loc="left", fontsize=10.5)
ax.legend(fontsize=8, loc="upper right"); ax.grid(alpha=0.15, lw=0.6)

# (b) t_dep distributions vs the SIMBA SF law and the observed
ax = axs[0, 1]
bins = np.linspace(8.0, 10.6, 27)
lti = np.log10(_col(_TQ, "tdep_gyr_inst") * 1e9)
ax.axvspan(np.log10(_tdep_sflaw_gyr(NH_SF[1]) * 1e9), np.log10(_tdep_sflaw_gyr(NH_SF[0]) * 1e9), color="#7570b3", alpha=0.12, lw=0,
           label=rf"SIMBA SF law $t_{{\rm ff}}/\epsilon_{{\rm ff}}$, $n_{{\rm H}}$ = {NH_SF[1]:g}–{NH_SF[0]:g} cm$^{{-3}}$ ({_tdep_sflaw_gyr(NH_SF[1]):.1f}–{_tdep_sflaw_gyr(NH_SF[0]):.1f} Gyr)")
ax.hist(ltq[np.isfinite(ltq)], bins=bins, histtype="step", lw=2.0, color="#1f1f1f", label=f"quenched, {STAGE_LABEL[FAIR_STAGE]}, 100 Myr SFR (N={int(np.isfinite(ltq).sum())})")
ax.hist(lti[np.isfinite(lti)], bins=bins, histtype="step", lw=1.6, ls="--", color="#1f1f1f", label=f"quenched, {STAGE_LABEL[FAIR_STAGE]}, instantaneous SFR (N={int(np.isfinite(lti).sum())})")
ax.hist(lts[np.isfinite(lts)], bins=bins, histtype="stepfilled", color="0.75", alpha=0.6, lw=0, label=f"SF controls, 100 Myr SFR (N={int(np.isfinite(lts).sum())})")
ax.hist(_lo, bins=bins, histtype="step", lw=2.2, color=C_OBS, label=f"ALMA-C11 QGs (N={len(_lo)}; $\\alpha_{{\\rm CO}}$ = 4.36, $R_{{31}}$ = 0.5)")
ax.axvline(np.log10(kl.tdep_ms_gyr(Z_OBS) * 1e9), color="0.3", ls=":", lw=1.0, label=f"Tacconi+18 MS at z={Z_OBS}")
ax.set_xlim(8.2, 10.6); ax.set_ylim(0, ax.get_ylim()[1] * 1.5); ax.set_xlabel(r"$\log(t_{\rm dep}/{\rm yr})$"); ax.set_ylabel("galaxies")
ax.set_title(r"(b) $t_{\rm dep}$: SIMBA's star-formation law fixes it at $t_{\rm ff}/\epsilon_{\rm ff}$ wherever there is H$_2$", loc="left", fontsize=10.5)
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.15, lw=0.6)
print(f"log t_dep[yr] medians — Q {FAIR_STAGE} 100 Myr: {np.nanmedian(ltq):.2f}, instantaneous: {np.nanmedian(lti):.2f}; SF: {np.nanmedian(lts):.2f}; observed: {np.median(_lo):.2f}")

# (c) the same Sigma_H2 against an H2-based relation
ax = axs[1, 0]
items_c = [_rel_line(ax, "B08", band=True), _rel_line(ax, "K98", band=False, lw=1.0, alpha=0.5)]
cq, it = _region_scatter(ax, xq, yq, ulq, "B08", show_sf=(xs_, ys_)); items_c += it
FAIR_ROWS.append(_count_row("c", "Sigma_H2", "B08", "Q", cq))
FAIR_ROWS.append(_count_row("c", "Sigma_H2", "B08", "SF", _region_scatter(plt.figure().gca(), xs_, ys_, uls, "B08")[0])); plt.close()
_oreg, _ = ks_region(OBS_S["logSigmaH2"].values, OBS_S["logSigmaSFR"].values, np.zeros(len(OBS_S), bool), ref="B08")
FAIR_ROWS.append(_count_row("c", "Sigma_H2", "B08", "obs", {c: (int((_oreg == c).sum()), 0) for c in REGION_ORDER}))
items_c += overlay_obs(ax, annotate=False)
ks_axes(ax, FIDUCIAL_AP)
ax.set_title(r"(c) H$_2$ against the H$_2$ relation Bigiel+08 ($\pm$0.20 dex); the total-gas K98 thin for reference", loc="left", fontsize=10.5)
ax.legend([h for h, _ in items_c], [l for _, l in items_c], loc="lower right", fontsize=7.8, frameon=False,
          title=f"Q at the {STAGE_LABEL[FAIR_STAGE]}; observed vs B08: " + ", ".join(f"{c} {int((_oreg == c).sum())}" for c in REGION_ORDER[:3]), title_fontsize=8)

# (d) total neutral gas against the total-gas relations
ax = axs[1, 1]
items_d = [_rel_line(ax, "K98", band=True), _rel_line(ax, "RK19", band=False, lw=1.0, alpha=0.7)]
cq, it = _region_scatter(ax, gq, yq, ulq, "K98", show_sf=(gs, ys_)); items_d += it
FAIR_ROWS.append(_count_row("d", "Sigma_HI+H2", "K98", "Q", cq))
FAIR_ROWS.append(_count_row("d", "Sigma_HI+H2", "K98", "SF", _region_scatter(plt.figure().gca(), gs, ys_, uls, "K98")[0])); plt.close()
FAIR_ROWS.append(_count_row("d", "Sigma_HI+H2", "RK19", "Q", _region_scatter(plt.figure().gca(), gq, yq, ulq, "RK19")[0])); plt.close()
FAIR_ROWS.append(_count_row("d", "Sigma_HI+H2", "RK19", "SF", _region_scatter(plt.figure().gca(), gs, ys_, uls, "RK19")[0])); plt.close()
for _, r in OBS_S.iterrows():                                  # observed: H2 only -> lower limits on Sigma_gas (arrows to the right)
    x0, y0 = float(r["logSigmaH2"]), float(r["logSigmaSFR"])
    ax.annotate("", xy=(x0 + 0.35, y0), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", color=C_OBS_EDGE, lw=1.1, mutation_scale=9), zorder=8)
    ax.scatter(x0, y0, s=90, marker="o", facecolors="white", edgecolors=C_OBS_EDGE, linewidths=1.3, zorder=9)
    ax.scatter(x0, y0, s=30, marker="o", c=C_OBS, edgecolors="none", zorder=10)
items_d.append((Line2D([], [], marker="o", ls="", ms=9, mfc="white", mec=C_OBS_EDGE), r"ALMA-C11 QGs: $\Sigma_{\rm H_2}$ only $\Rightarrow$ lower limit on $\Sigma_{\rm gas}$"))
ks_axes(ax, FIDUCIAL_AP)
ax.set_xlabel(r"$\log(\Sigma_{\rm HI+H_2}/M_\odot\,{\rm pc}^{-2})$  [neutral gas inside " + AP_TITLE[FIDUCIAL_AP] + rf", $\times{HE_FACTOR:g}$ He]")
_dg = np.nanmedian(gq - xq)
ax.set_title(rf"(d) total neutral gas against K98 (its own calibration; median $\Sigma_{{\rm gas}}/\Sigma_{{\rm H_2}}$ = {10 ** _dg:.2f})", loc="left", fontsize=10.5)
ax.legend([h for h, _ in items_d], [l for _, l in items_d], loc="lower right", fontsize=7.8, frameon=False, title=f"Q at the {STAGE_LABEL[FAIR_STAGE]}", title_fontsize=8)

fig.suptitle("Is the offset from the KS relation a Σ-definition effect? — SIMBA-25 quenched galaxies at the track end vs ALMA-C11   "
             f"[{AP_TITLE[FIDUCIAL_AP]}; 100 Myr archaeological SFR]", y=0.995)
plt.tight_layout(rect=(0, 0, 1, 0.975))
fig.savefig(os.path.join(FIGDIR, f"ks_fair_comparison_{FIDUCIAL_AP}.png"), dpi=160, bbox_inches="tight"); plt.show()

# the K98 rows (H2 vs the total-gas relation, for the record) + table
FAIR_ROWS.insert(0, _count_row("0", "Sigma_H2", "K98", "Q", _region_scatter(plt.figure().gca(), xq, yq, ulq, "K98")[0])); plt.close()
FAIR_ROWS.insert(1, _count_row("0", "Sigma_H2", "K98", "SF", _region_scatter(plt.figure().gca(), xs_, ys_, uls, "K98")[0])); plt.close()
_oreg, _ = ks_region(OBS_S["logSigmaH2"].values, OBS_S["logSigmaSFR"].values, np.zeros(len(OBS_S), bool), ref="K98")
FAIR_ROWS.insert(2, _count_row("0", "Sigma_H2", "K98", "obs", {c: (int((_oreg == c).sum()), 0) for c in REGION_ORDER}))
FAIR = Table(rows=FAIR_ROWS)
FAIR.to_pandas().to_csv(os.path.join(KSDIR, "ks_fair_comparison.csv"), index=False)
print(f"{'panel':>5s} {'x':12s} {'ref':5s} {'pop':4s} {'N':>4s}  below  within  above  undef")
for r in FAIR:
    print(f"{r['panel']:>5s} {r['x']:12s} {r['relation']:5s} {r['pop']:4s} {r['n']:4d}  "
          + "  ".join(f"{100 * r[f'n_{c}'] / r['n']:4.0f}%" if r["n"] else "   -" for c in REGION_ORDER))
print(f"-> {os.path.join(KSDIR, 'ks_fair_comparison.csv')}")


In [ ]:
# ── Part 4 summary — per (stage, aperture): N, censoring, medians, offsets from the relations ──
_rows = []
for ap in AP_LABELS:
    T = _sel_ap(ap, "Q")
    stg, ul = _s(T["stage"]), np.asarray(T["is_ul"], bool)
    x, y = np.asarray(T["logSigmaH2"], float), np.asarray(T["logSigmaSFR_obs"], float)
    td, rk = np.asarray(T["tdep_gyr"], float), np.asarray(T["ap_kpc"], float)
    for st in STAGES_PLOT + ["end"]:
        m_all = stg == st
        m = m_all & np.isfinite(x) & np.isfinite(y) & ~ul
        r = dict(aperture=ap, stage=st, n_rows=int(m_all.sum()), n_sigma_h2=int((m_all & np.isfinite(x)).sum()),
                 n_both=int(m.sum()), n_sfr_ul=int((m_all & np.isfinite(x) & ul).sum()),
                 f_r50_undefined=float(np.mean(~np.isfinite(rk[m_all]))) if m_all.any() else np.nan)
        for key, arr in (("logSigmaH2", x), ("logSigmaSFR", y), ("log_tdep_gyr", np.log10(np.where(td > 0, td, np.nan))),
                         ("r_kpc", rk)):
            v = arr[m]
            v = v[np.isfinite(v)]
            r[key + "_med"] = float(np.median(v)) if v.size else np.nan
            r[key + "_p16"] = float(np.percentile(v, 16)) if v.size else np.nan
            r[key + "_p84"] = float(np.percentile(v, 84)) if v.size else np.nan
        for key in kl.RELATIONS:
            d = y[m] - kl.relation_y(key, x[m])
            r["dlog_" + key + "_med"] = float(np.median(d)) if d.size else np.nan
        _rows.append(r)
SUMMARY = Table(rows=_rows)
write_table(SUMMARY, os.path.join(KSDIR, "ks_stage_summary.fits"))
SUMMARY.to_pandas().to_csv(os.path.join(KSDIR, "ks_stage_summary.csv"), index=False)
_show = SUMMARY[_s(SUMMARY["aperture"]) == FIDUCIAL_AP]
print(f"fiducial aperture {FIDUCIAL_AP} (Q galaxies; medians over rows with both Sigma finite and SFR>0):")
for r in _show:
    print(f"  {r['stage']:12s} N={r['n_both']:3d} (+{r['n_sfr_ul']:3d} SFR UL, R50 undef {100*r['f_r50_undefined']:4.0f}%)  "
          f"logSigH2={r['logSigmaH2_med']:+.2f} [{r['logSigmaH2_p16']:+.2f},{r['logSigmaH2_p84']:+.2f}]  "
          f"logSigSFR={r['logSigmaSFR_med']:+.2f}  log tdep={r['log_tdep_gyr_med']:+.2f}  "
          f"R50={r['r_kpc_med']:.2f} kpc  dlog B08={r['dlog_B08_med']:+.2f} K98={r['dlog_K98_med']:+.2f}")
print("\nobserved ALMA-C11 (CO-detected with size):")
print(OBS_S[["id", "z", "logSigmaH2", "logSigmaSFR", "sigma_is_ll", "r_kpc", "tdep_Gyr", "dlog_B08", "dlog_K98"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Part 5 — who ends up below, on, or above the relation? Evolutionary properties of the KS regions

Part 4 sorts the quenched galaxies by the **region of their track end** w.r.t. the reference relation (`ks_region_anchor`:
below / on / above the ±0.20 dex band of Bigiel+08 — 103 / 94 / 65 galaxies in the real m25 run, 3 undetermined). Here those
three groups are followed through their whole histories, in the same four layouts every time: the three anchor-redshift
ranges of the track figures **and** all anchors together.

* **Clock**: $t - t_{\rm QT}$ (quench end = 0). Every history (one row per snapshot, `ks_track_histories.fits`, Part 5a) is
  resampled on a −2 … +3 Gyr grid and the per-group **median (16–84 %)** is drawn where ≥ `NMIN_GRID` galaxies cover the grid
  point — coverage at large $t - t_{\rm QT}$ shrinks to the early quenchers (the histories end at the anchor). Galaxies without a
  detected quench event (`no_event`, 28/266) have no clock and appear only in the clock-free panels.
* **What is followed**: quench timing (Figure 4: $z_{\rm QT}$, $\tau_{\rm q}$, sSFR-peak → QT, time quenched at the track end);
  the AGN side (Figure 5: coupling class and $x_{\rm str}$ of the selection, Eddington ratio and $M_{\rm BH}/M_\star$ from the BH
  histories); the **dust-fraction / stellar-age plane** (Figure 6: catalogue $M_{\rm dust}/M_\star$ vs the caesar mass-weighted
  stellar age, both per snapshot); **rotation** (Figure 7: caesar $\kappa_{\rm rot}$ of the stars and of the member gas, stellar
  B/T, read from the catalogue of every snapshot through the progenitor index; the H$_2$-weighted $\kappa_{\rm rot}$ of the
  powderday cutouts at the anchor when `tables/annulus_kinematics.fits` exists); the **molecular content and extent**
  (Figure 8: $M_{\rm H_2}/M_\star$, the catalogue gas half-mass radius and its ratio to the stellar one on the clock; the
  face-on member $R_{50}$(H$_2$) of Part 3 and $R_{50}$(H$_2$)/$R_{50}(\star)$ at the drawn stages).
* **Controls**: the mass-matched SF partners at their anchor (grey median + 16–84 % band, or dashed ECDF); they are not in the
  history files, so they have no clock.
* Tables → `ks_region_properties.csv` (per z column × region: sizes, class fractions, medians at the track end, two-sample KS
  p-values below-vs-above and below-vs-on), `ks_region_fdust_age_tracks.csv`, `ks_region_h2_extent.csv`.

In [ ]:
# ── Part 5a — continuous histories of the quenched sample (cached) ──
# One row per (anchor, galaxy, history snapshot): the history-file properties (masses, SFR, mass-weighted stellar age,
# catalogue half-mass radii), the BH history (M_BH, Eddington ratio, accretion rate) and — through the progenitor index —
# the caesar catalogue's structure at that snapshot (kappa_rot and B/T of the stars and of the member gas, central flag,
# 100 Myr catalogue SFR, stellar metallicity). The SF controls are not in the history files: they get ONE row each (their
# anchor snapshot, everything read from the anchor catalogue). Catalogue/history radii are COMOVING kpc -> stored in
# proper kpc (x a). Rebuilt when the cache is absent, when OVERWRITE_HISTORIES, or when it lacks a current column.
HIST_FITS = os.path.join(KSDIR, "ks_track_histories.fits")
OVERWRITE_HISTORIES = False
_H_KEYS = {"mstar": "masses.stellar", "sfr": "sfr", "mh2": "masses.H2", "mhi": "masses.HI", "mgas": "masses.gas",
           "mdust": "masses.dust", "age_mw": "ages.mass_weighted", "r50gas": "radii.gas_half_mass",
           "r50star": "radii.stellar_half_mass", "ngas": "ngas", "nstar": "nstar"}
_BH_KEYS = {"bh_mass": "bh_mass", "bh_fedd": "bh_fedd", "bh_mdot": "bh_mdot"}
_CAT_KEYS = {"kappa_star": "dicts/rotation.stellar_kappa_rot", "kappa_gas": "dicts/rotation.gas_kappa_rot",
             "bt_star": "dicts/rotation.stellar_BoverT", "bt_gas": "dicts/rotation.gas_BoverT",
             "zstar": "dicts/metallicities.stellar", "sfr100_cat": "sfr_100", "central": "central"}
_CAT_ALL = {"mstar": "dicts/masses.stellar", "sfr": "sfr", "mh2": "dicts/masses.H2", "mhi": "dicts/masses.HI",
            "mgas": "dicts/masses.gas", "mdust": "dicts/masses.dust", "age_mw": "dicts/ages.mass_weighted",
            "r50gas": "dicts/radii.gas_half_mass", "r50star": "dicts/radii.stellar_half_mass", "ngas": "ngas", "nstar": "nstar",
            "bh_mass": "dicts/masses.bh", "bh_fedd": "bh_fedd", "bh_mdot": "bhmdot", **_CAT_KEYS}
HIST_COLS = ["anchor_z", "anchor_snap", "gal_id", "gkey", "pop", "snap", "gx", "z", "t_gyr"] + list(_H_KEYS) + list(_BH_KEYS) + list(_CAT_KEYS)
COMOVING_COLS = ("r50gas", "r50star")      # caesar `radii.*` are kpccm (the history builder copied them as such)


def read_catalog_props(snap, gxs, keys):
    """{key: values at galaxy indices `gxs`} read directly from the caesar file of `snap` (galaxy_data/<dataset>;
    GroupID == row index); NaN where a dataset is absent, the file is missing, or gx is out of range."""
    gxs = np.asarray(gxs, int)
    out = {k: np.full(len(gxs), np.nan) for k in keys}
    path = sim.get_caesar_file(snap)
    if not os.path.exists(path):
        warnings.warn(f"no caesar file for snap {snap}: catalogue columns stay NaN there")
        return out
    with h5py.File(path, "r") as f:
        for k, ds in keys.items():
            if ("galaxy_data/" + ds) not in f:
                continue
            arr = np.asarray(f["galaxy_data/" + ds][:], float)
            ok = (gxs >= 0) & (gxs < len(arr))
            out[k][ok] = arr[gxs[ok]]
    return out


HIST = None
if os.path.exists(HIST_FITS) and not OVERWRITE_HISTORIES:
    HIST = Table.read(HIST_FITS)
    _lack = [c for c in HIST_COLS if c not in HIST.colnames]
    if _lack:
        print(f"cached history table lacks {_lack} -> rebuilt now")
        HIST = None
    else:
        print(f"cached ({len(HIST)} rows) -> {HIST_FITS}   (OVERWRITE_HISTORIES=True rebuilds)")
if HIST is None:
    SEL, SNAPS, IDS = load_selection()
    POP = _s(SEL["pop"])
    _parts = []
    for _zt, A in ANCHORS.items():
        _inA = SNAPS == A["snap"]
        _q, _sf = SEL[_inA & (POP == "Q")], SEL[_inA & (POP == "SF")]
        if len(_q):
            H = load_anchor_history(A)
            _g2c = {int(g): j for j, g in enumerate(H["galaxy_ids"])}
            _cols = np.array([_g2c[int(g)] for g in _q["gal_id"]], int)          # Part 1 already verified presence
            PIDX = build_prog_index(A, H["galaxy_ids"], H["snaps_arr"])
            _bh = {}
            _bhp = os.path.join(SFHDIR, f"bh_history_anchor_{A['tag']}.hdf5")
            if os.path.exists(_bhp):
                with h5py.File(_bhp, "r") as f:
                    _bh = {k: f[v][:] for k, v in _BH_KEYS.items() if v in f}
            else:
                print(f"  [{A['tag']}] no BH history ({os.path.basename(_bhp)}): BH columns NaN")
            ns = len(H["snaps_arr"])
            R, Qi = np.meshgrid(np.arange(ns), np.arange(len(_q)), indexing="ij")   # history row x selection index
            C = _cols[Qi]
            gx = PIDX[R, C]
            keep = gx >= 0                                                        # tracked progenitor at that snapshot
            R, Qi, C, gx = R[keep], Qi[keep], C[keep], gx[keep]
            D = dict(anchor_z=np.full(len(R), float(_zt)), anchor_snap=np.full(len(R), int(A["snap"])),
                     gal_id=np.asarray(_q["gal_id"], int)[Qi], pop=np.full(len(R), "Q"),
                     snap=np.asarray(H["snaps_arr"], int)[R], gx=gx.astype(int),
                     z=np.asarray(H["redshift"], float)[R], t_gyr=np.asarray(H["t_cosmic_yr"], float)[R] / 1e9)
            for k, pk in _H_KEYS.items():
                D[k] = np.asarray(H["P"][pk], float)[R, C] if pk in H["P"] else np.full(len(R), np.nan)
            for k in _BH_KEYS:
                D[k] = np.asarray(_bh[k], float)[R, C] if k in _bh else np.full(len(R), np.nan)
            for k in _CAT_KEYS:
                D[k] = np.full(len(R), np.nan)
            _parts.append(Table(D))
            print(f"  [{A['tag']}] {len(_q)} Q galaxies -> {len(R)} history rows ({ns} snapshots)")
            del H, PIDX
            gc.collect()
        if INCLUDE_SF_CONTROL and len(_sf):
            _gx = np.asarray(_sf["gal_id"], int)
            D = dict(anchor_z=np.full(len(_sf), float(_zt)), anchor_snap=np.full(len(_sf), int(A["snap"])),
                     gal_id=_gx, pop=np.full(len(_sf), "SF"), snap=np.full(len(_sf), int(A["snap"])), gx=_gx,
                     z=np.full(len(_sf), float(A["z"])), t_gyr=np.full(len(_sf), float(COSMO.age(A["z"]).value)))
            D.update(read_catalog_props(A["snap"], _gx, _CAT_ALL))
            _parts.append(Table(D))
    HIST = vstack(_parts)
    # catalogue structure of the tracked progenitors: one read per snapshot
    _isQ = _s(HIST["pop"]) == "Q"
    _snapc, _gxc = np.asarray(HIST["snap"], int), np.asarray(HIST["gx"], int)
    for _sn in np.unique(_snapc[_isQ]):
        _i = np.where(_isQ & (_snapc == _sn))[0]
        for k, v in read_catalog_props(_sn, _gxc[_i], _CAT_KEYS).items():
            HIST[k][_i] = v
    _a = 1.0 / (1.0 + np.asarray(HIST["z"], float))
    for k in COMOVING_COLS:
        HIST[k] = np.asarray(HIST[k], float) * _a                              # -> proper kpc
    HIST["gkey"] = np.array([f"{a}_{g}" for a, g in zip(HIST["anchor_snap"], HIST["gal_id"])])
    HIST = HIST[HIST_COLS]
    write_table(HIST, HIST_FITS)
    print(f"wrote {len(HIST)} rows -> {HIST_FITS}")

_hp = _s(HIST["pop"])
_hk = np.asarray(HIST["gkey"]).astype(str)
print(f"histories: {len(np.unique(_hk[_hp == 'Q']))} Q galaxies x {len(np.unique(np.asarray(HIST['snap'])[_hp == 'Q']))} snapshots "
      f"({int((_hp == 'Q').sum())} rows), SF anchor rows: {int((_hp == 'SF').sum())}")
print("finite fraction per column (Q rows): " + ", ".join(
    f"{c}={np.isfinite(np.asarray(HIST[c], float)[_hp == 'Q']).mean():.2f}"
    for c in ("age_mw", "mdust", "mh2", "r50gas", "bh_fedd", "kappa_star", "kappa_gas", "bt_star")))

In [ ]:
# ── Part 5b — the groups (region of the track end w.r.t. KS_REF), the quench clock, grid medians, panel helpers ──
# Every Part 5 figure has FOUR columns: the three anchor-redshift ranges of Z_PANELS and all anchors together (Z_ALL);
# the groups are the three regions of the track end (Part 4c `ks_region_anchor`, the `ksreg` scheme); the common clock
# is t - t_QT, so galaxies without a detected quench event (agn_class = no_event) enter only the clock-free panels.
REGION_GROUPS = ["below", "within", "above"]
REGION_SHORT = {"below": f"below {KS_REF}", "within": f"on {KS_REF}", "above": f"above {KS_REF}", "undef": "undetermined"}
Z_COLS = [(lo, hi, t) for (lo, hi, _), t in zip(Z_PANELS + Z_ALL, (r"anchors $z\leq0.5$ (ALMA-C11 range)", r"anchors $0.7\leq z\leq1$",
                                                             r"anchors $1.15\leq z\leq2$", r"all anchors $0.3\leq z\leq2$"))]
ZCOL_NAME = ["z<=0.5", "0.7-1", "1.15-2", "all z"]         # plain names for the printed tables
DT_GRID = np.arange(-2.0, 3.001, 0.25)     # quench clock t - t_QT [Gyr] on which the histories are resampled
NMIN_GRID = 5                              # galaxies covering a grid point needed for its median
NMIN_ECDF = 4                              # galaxies needed for an ECDF / a KS test sample
CLASS_ORDER = ["weak", "intermediate", "strong", "no_event"]
CLASS_COLOR = {"weak": "#1b9e77", "intermediate": "#7570b3", "strong": "#d95f02", "no_event": "0.65"}
C_SF = "0.5"
GROUP_DEF = (rf"groups = region of the track end w.r.t. {KS_REF} ($\pm${KS_BAND:.2f} dex band; {END_DEF}); "
             r"clock = $t - t_{\rm QT}$; grey = SF controls at their anchor (median, 16–84 %)")
try:
    from scipy.stats import ks_2samp as _ks2

    def ks_p(a, b):
        """Two-sample KS p-value (NaN when either sample has < NMIN_ECDF values)."""
        a, b = [np.asarray(v, float)[np.isfinite(np.asarray(v, float))] if v is not None else np.zeros(0) for v in (a, b)]
        return float(_ks2(a, b).pvalue) if (len(a) >= NMIN_ECDF and len(b) >= NMIN_ECDF) else np.nan
except ImportError:                                            # no scipy in the kernel: the figures still draw
    def ks_p(a, b):
        return np.nan

# ── one row per quenched galaxy (ANCHOR_PROPS order): group, quench clock, end-row structure ──
_EA = EPOCHS[(_s(EPOCHS["stage"]) == "anchor") & (_s(EPOCHS["pop"]) == "Q")]
_ea = {f"{a}_{g}": i for i, (a, g) in enumerate(zip(np.asarray(_EA["anchor_snap"]), np.asarray(_EA["gal_id"])))}
_ia = np.array([_ea[g] for g in np.asarray(ANCHOR_PROPS["gkey"]).astype(str)], int)


def _anc(name, dtype=float):
    return np.asarray(_EA[name])[_ia].astype(dtype)


def _epoch_col(stage, name):
    """`name` of the galaxy's `stage` row of EPOCHS (even without a reduced file), aligned to GALS (NaN if none)."""
    E = EPOCHS[(_s(EPOCHS["stage"]) == stage) & (_s(EPOCHS["pop"]) == "Q")]
    d = dict(zip((f"{a}_{g}" for a, g in zip(np.asarray(E["anchor_snap"]), np.asarray(E["gal_id"]))), np.asarray(E[name], float)))
    return np.array([d.get(g, np.nan) for g in np.asarray(ANCHOR_PROPS["gkey"]).astype(str)])


def _stage_col(stage, name, ap=FIDUCIAL_AP):
    """`name` of the galaxy's measured `stage` row of TRACKS (aperture `ap`), aligned to GALS (NaN if none)."""
    T = TRACKS[(_s(TRACKS["stage"]) == stage) & (_s(TRACKS["pop"]) == "Q") & (_s(TRACKS["ap_label"]) == ap)]
    d = dict(zip(np.asarray(T["gkey"]).astype(str), np.asarray(T[name], float)))
    return np.array([d.get(g, np.nan) for g in np.asarray(ANCHOR_PROPS["gkey"]).astype(str)])


GALS = Table(dict(gkey=np.asarray(ANCHOR_PROPS["gkey"]).astype(str), anchor_z=np.asarray(ANCHOR_PROPS["anchor_z"], float),
                  region=np.asarray(ANCHOR_PROPS["ks_region_anchor"]).astype(str), agn_class=np.asarray(ANCHOR_PROPS["agn_class"]).astype(str),
                  log_mstar_anchor=np.asarray(ANCHOR_PROPS["log_mstar_anchor"], float),
                  xstr_quench=_anc("xstr_quench"), t_qt_gyr=_anc("t_qt_gyr"), t_sft_gyr=_anc("t_sft_gyr"),
                  tau_q_gyr=_anc("tau_q_gyr"), z_qt=_anc("z_qt"), t_anchor_gyr=_anc("t_stage_gyr"), n_events=_anc("n_events", int),
                  t_sfpeak_gyr=_epoch_col("sf_peak", "t_stage_gyr"),
                  t_end_gyr=_epoch_col("end", "t_stage_gyr"), dt_end_gyr=_epoch_col("end", "dt_from_qt_gyr"),
                  log_fh2_anchor=np.asarray(ANCHOR_PROPS["log_fh2_anchor"], float)))
GALS["has_clock"] = np.isfinite(np.asarray(GALS["t_qt_gyr"], float))
for _st in STAGES_DRAW:
    GALS[f"snap_{_st}"] = _epoch_col(_st, "snap")
    GALS[f"r50h2_{_st}"] = _stage_col(_st, "ap_kpc")          # face-on member H2 half-mass radius [proper kpc] (R50_H2 row)
    GALS[f"r50star_{_st}"] = _stage_col(_st, "r50_star")      # face-on member stellar half-mass radius [proper kpc]


def members(zlo, zhi, region=None, need_clock=False):
    """Boolean mask over GALS: anchor z in (zlo, zhi], optional region, optional detected quench event."""
    az = np.asarray(GALS["anchor_z"], float)
    m = (az > zlo) & (az <= zhi)
    if region is not None:
        m &= np.asarray(GALS["region"]).astype(str) == region
    if need_clock:
        m &= np.asarray(GALS["has_clock"], bool)
    return m


# ── the histories on the clock ──
_hk = np.asarray(HIST["gkey"]).astype(str)
_tq = dict(zip(np.asarray(GALS["gkey"]), np.asarray(GALS["t_qt_gyr"], float)))
HIST["dt_qt_gyr"] = np.asarray(HIST["t_gyr"], float) - np.array([_tq.get(g, np.nan) for g in _hk])
_o = np.argsort(_hk, kind="stable")
_ug, _st0, _cnt = np.unique(_hk[_o], return_index=True, return_counts=True)
HROWS = {g: _o[s:s + c] for g, s, c in zip(_ug, _st0, _cnt)}                       # gkey -> its HIST row indices
HKEY = {(g, int(s)): i for i, (g, s) in enumerate(zip(_hk, np.asarray(HIST["snap"])))}   # (gkey, snap) -> HIST row


def hist_values(col):
    """HIST column `col` as float, or `col` itself when it is already an array (derived quantity)."""
    return np.asarray(HIST[col], float) if isinstance(col, str) else np.asarray(col, float)


def hist_at_stage(stage, col):
    """`col` of the HIST row at the galaxy's `stage` snapshot, aligned to GALS (NaN without that row)."""
    v = hist_values(col)
    sn = np.asarray(GALS[f"snap_{stage}"], float)
    return np.array([v[HKEY[(g, int(s))]] if (np.isfinite(s) and (g, int(s)) in HKEY) else np.nan
                     for g, s in zip(np.asarray(GALS["gkey"]), sn)])


def grid_tracks(gkeys, col):
    """(n_gal, n_grid) array: `col` of every galaxy's history resampled onto DT_GRID (NaN outside its coverage)."""
    v, dt = hist_values(col), np.asarray(HIST["dt_qt_gyr"], float)
    out = np.full((len(gkeys), len(DT_GRID)), np.nan)
    for i, g in enumerate(gkeys):
        r = HROWS.get(g)
        if r is not None:
            out[i] = kl.interp_track(dt[r], v[r], DT_GRID)
    return out


def sf_band(ax, zlo, zhi, col):
    """Median (dashed) and 16-84 % band of `col` over the SF controls whose anchor lies in (zlo, zhi]."""
    m = (_s(HIST["pop"]) == "SF") & (np.asarray(HIST["anchor_z"], float) > zlo) & (np.asarray(HIST["anchor_z"], float) <= zhi)
    v = hist_values(col)[m]
    v = v[np.isfinite(v)]
    if len(v) >= NMIN_ECDF:
        ax.axhspan(np.percentile(v, 16), np.percentile(v, 84), color=C_SF, alpha=0.13, lw=0, zorder=0)
        ax.axhline(np.median(v), color=C_SF, lw=1.0, ls="--", zorder=1)
    return v


def draw_clock(ax, zlo, zhi, col, ylabel, band=True, regions=REGION_GROUPS, sf=True):
    """Median (16-84 % band) of `col` against t - t_QT for every region of the galaxies with anchor z in (zlo, zhi]
    and a detected quench event; the SF controls' anchor range as the grey band. Returns {region: grid_stats}."""
    res = {}
    for reg in regions:
        gk = np.asarray(GALS["gkey"])[members(zlo, zhi, reg, need_clock=True)]
        st = kl.grid_stats(grid_tracks(gk, col), NMIN_GRID) if len(gk) else None
        res[reg] = st
        if st is None or not np.isfinite(st["med"]).any():
            continue
        ok = np.isfinite(st["med"])
        n_at0 = int(st["n"][np.argmin(np.abs(DT_GRID))])
        ax.plot(DT_GRID[ok], st["med"][ok], color=REGION_COLOR[reg], lw=2.2, zorder=4,
                label=f"{REGION_SHORT[reg]} (N={len(gk)}; {n_at0} at QT)")
        if band:
            ax.fill_between(DT_GRID[ok], st["p16"][ok], st["p84"][ok], color=REGION_COLOR[reg], alpha=0.12, lw=0, zorder=2)
    if sf:
        sf_band(ax, zlo, zhi, col)
    ax.axvline(0, color="0.5", lw=0.8, ls=":")
    ax.set_xlim(DT_GRID[0], DT_GRID[-1]); ax.set_xlabel(r"$t - t_{\rm QT}$ [Gyr]"); ax.set_ylabel(ylabel)
    ax.grid(alpha=0.15, lw=0.6); ax.legend(fontsize=7.5, frameon=False)
    return res


def ecdf_panel(ax, zlo, zhi, values, xlabel, need_clock=False, regions=REGION_GROUPS, sf_values=None, xlim=None):
    """ECDFs of the per-galaxy array `values` (GALS order) for every region (median ticks at F = 0.5) and, optionally, of the
    SF controls (`sf_values`, any array); prints the KS p-value of below vs above. Returns {region: sample}."""
    vals = np.asarray(values, float)
    samples = {}
    for reg in regions:
        m = members(zlo, zhi, reg, need_clock) & np.isfinite(vals)
        if m.sum() < NMIN_ECDF:
            continue
        x, F = kl.ecdf(vals[m])
        samples[reg] = vals[m]
        ax.step(np.r_[x[0], x], np.r_[0, F], where="post", color=REGION_COLOR[reg], lw=2.0, zorder=4,
                label=f"{REGION_SHORT[reg]} (N={int(m.sum())}, med {np.median(vals[m]):.2f})")
        ax.plot([np.median(vals[m])], [0.5], marker="|", ms=13, mew=2.2, color=REGION_COLOR[reg], zorder=5)
    if sf_values is not None:
        sv = np.asarray(sf_values, float)
        sv = sv[np.isfinite(sv)]
        if len(sv) >= NMIN_ECDF:
            x, F = kl.ecdf(sv)
            ax.step(np.r_[x[0], x], np.r_[0, F], where="post", color=C_SF, lw=1.3, ls="--", zorder=3, label=f"SF controls (N={len(sv)})")
    p = ks_p(samples.get("below"), samples.get("above"))
    ax.set_ylim(0, 1.02); ax.set_ylabel("cumulative fraction"); ax.set_xlabel(xlabel)
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.grid(alpha=0.15, lw=0.6)
    ax.legend(fontsize=7.5, loc="lower right", frameon=False, title=f"KS below vs above: p = {p:.2g}" if np.isfinite(p) else None, title_fontsize=7.5)
    return samples


def region_grid(nrows, height=3.4):
    """Figure with `nrows` rows x the four z columns; column titles on the first row; returns (fig, axs)."""
    fig, axs = plt.subplots(nrows, len(Z_COLS), figsize=(4.7 * len(Z_COLS), height * nrows), squeeze=False)
    for j, (_, _, ttl) in enumerate(Z_COLS):
        axs[0, j].set_title(ttl, fontsize=9.5, loc="left", pad=16)
    return fig, axs


def letter(ax, i, j, text):
    ax.set_title(f"({'abcdefghij'[i]}{j + 1}) {text}", fontsize=9, loc="right", pad=3 if i else 4)


def finish_region_fig(fig, fname, title):
    fig.suptitle(f"{title}\n{GROUP_DEF}", y=0.995, fontsize=10.5)
    plt.tight_layout(rect=(0, 0, 1, 0.965))
    fig.savefig(os.path.join(FIGDIR, fname), dpi=150, bbox_inches="tight")
    plt.show()


print(f"{'column':10s} " + " ".join(f"{REGION_SHORT[r]:>12s}" for r in REGION_GROUPS) + f" {'undef':>6s} {'no clock':>9s}")
for (zlo, zhi, _), name in zip(Z_COLS, ZCOL_NAME):
    m = members(zlo, zhi)
    print(f"{name:10s} " + " ".join(f"{int(members(zlo, zhi, r).sum()):12d}" for r in REGION_GROUPS)
          + f" {int((m & (np.asarray(GALS['region']).astype(str) == 'undef')).sum()):6d} {int((m & ~np.asarray(GALS['has_clock'], bool)).sum()):9d}")
print(f"history coverage on the clock: {int(np.asarray(GALS['has_clock'], bool).sum())} galaxies with t_QT; grid {DT_GRID[0]:+.2f} … {DT_GRID[-1]:+.2f} Gyr "
      f"(step {DT_GRID[1] - DT_GRID[0]:.2f}); a grid median needs {NMIN_GRID} galaxies — coverage at large t - t_QT is limited to early quenchers")

In [ ]:
# ── Figure 4 — when and how fast did each group quench? (clock-free: distributions per region) ──
# (a) redshift of the quench end z_QT; (b) quench duration tau_q = t_QT - t_SFT; (c) time from the sSFR peak to QT (the
# whole decline); (d) time already spent quenched at the track end, t_end - t_QT. Galaxies without a detected event
# have none of these (no_event: counted in Figure 5a only).
_tau = np.asarray(GALS["tau_q_gyr"], float)
_decl = np.asarray(GALS["t_qt_gyr"], float) - np.asarray(GALS["t_sfpeak_gyr"], float)
TIMING_ROWS = [("z_qt", np.asarray(GALS["z_qt"], float), r"$z_{\rm QT}$ (quench end)", (0, 4.5)),
               ("log_tau_q", np.log10(np.where(_tau > 0, _tau, np.nan)), r"$\log(\tau_{\rm q}/{\rm Gyr})$  [$t_{\rm QT} - t_{\rm SFT}$]", (-2.2, 1.0)),
               ("t_decline", np.where(_decl > 0, _decl, np.nan), r"$t_{\rm QT} - t_{\rm sSFR\,peak}$ [Gyr]", (0, 8)),
               ("dt_end", np.asarray(GALS["dt_end_gyr"], float), r"$t_{\rm end} - t_{\rm QT}$ [Gyr]  (time quenched at the track end)", (-0.5, 8))]
fig, axs = region_grid(len(TIMING_ROWS), height=3.3)
TIMING_SAMPLES = {}
for i, (key, vals, lab, xlim) in enumerate(TIMING_ROWS):
    for j, (zlo, zhi, _) in enumerate(Z_COLS):
        TIMING_SAMPLES[(key, j)] = ecdf_panel(axs[i, j], zlo, zhi, vals, lab, need_clock=True, xlim=xlim)
        letter(axs[i, j], i, j, key)
finish_region_fig(fig, f"ks_regions_timing_{FIDUCIAL_AP}.png",
                  "SIMBA-25 quenched galaxies by KS region — quench timing (quench end, duration, decline, time quenched at the track end)")
for key, vals, lab, _ in TIMING_ROWS:
    print(f"{key:10s} medians " + " | ".join(
        f"{ZCOL_NAME[j]:7s}: " + " ".join(f"{r[:3]} {np.median(TIMING_SAMPLES[(key, j)][r]):5.2f}" if r in TIMING_SAMPLES[(key, j)] else f"{r[:3]}   n/a"
                                                 for r in REGION_GROUPS) for j in range(len(Z_COLS))))

In [ ]:
# ── Figure 5 — the AGN side of each group ──
# (a) composition in AGN coupling class during the quenching phase (selection table; no_event = no detected quench);
# (b) coupling strength x_str of the quench window; (c) black-hole Eddington ratio along the clock (BH history; galaxies
# without a BH -> NaN; f_Edd floored at 1e-5 on the log axis); (d) log M_BH/M* along the clock.
_bhm, _fedd, _mst = (np.asarray(HIST[c], float) for c in ("bh_mass", "bh_fedd", "mstar"))
with np.errstate(divide="ignore", invalid="ignore"):
    LOG_FEDD = np.where(_bhm > 0, np.log10(np.clip(_fedd, 1e-5, None)), np.nan)
    LOG_MBH_MSTAR = np.where((_bhm > 0) & (_mst > 0), np.log10(_bhm / _mst), np.nan)
_cls = np.asarray(GALS["agn_class"]).astype(str)

fig, axs = region_grid(4, height=3.4)
AGN_FRAC = []
for j, (zlo, zhi, ttl) in enumerate(Z_COLS):
    ax = axs[0, j]
    for k, reg in enumerate(REGION_GROUPS):
        m = members(zlo, zhi, reg)
        n = int(m.sum())
        left = 0.0
        for c in CLASS_ORDER:
            f = float((m & (_cls == c)).sum()) / n if n else 0.0
            ax.barh(k, f, left=left, color=CLASS_COLOR[c], edgecolor="white", lw=0.6, height=0.62)
            if f >= 0.08:
                ax.text(left + f / 2, k, f"{100 * f:.0f}%", ha="center", va="center", fontsize=7.5, color="white" if c != "no_event" else "0.2")
            left += f
            AGN_FRAC.append(dict(z_lo=zlo, z_hi=zhi, region=reg, agn_class=c, n=int((m & (_cls == c)).sum()), n_region=n, fraction=f))
        ax.text(1.02, k, f"N={n}", va="center", fontsize=8)
    ax.set_yticks(range(len(REGION_GROUPS))); ax.set_yticklabels([REGION_SHORT[r] for r in REGION_GROUPS], fontsize=8.5)
    ax.set_xlim(0, 1.18); ax.set_xlabel("fraction of the region"); ax.invert_yaxis()
    for k, r in enumerate(REGION_GROUPS):
        ax.get_yticklabels()[k].set_color(REGION_COLOR[r])
    if j == 0:
        ax.legend([Patch(color=CLASS_COLOR[c]) for c in CLASS_ORDER], CLASS_ORDER, fontsize=7.5, ncol=2, frameon=False, loc="lower left",
                  bbox_to_anchor=(0.0, -0.62), title="AGN coupling class during the quench phase", title_fontsize=8)
    letter(ax, 0, j, "AGN class composition")
    ecdf_panel(axs[1, j], zlo, zhi, np.asarray(GALS["xstr_quench"], float), r"$x_{\rm str}$ (coupling strength over [SFT, QT])", need_clock=True)
    letter(axs[1, j], 1, j, "coupling strength")
    draw_clock(axs[2, j], zlo, zhi, LOG_FEDD, r"$\log f_{\rm Edd}$ (BH history, floor $10^{-5}$)")
    letter(axs[2, j], 2, j, "Eddington ratio on the clock")
    draw_clock(axs[3, j], zlo, zhi, LOG_MBH_MSTAR, r"$\log(M_{\rm BH}/M_\star)$")
    letter(axs[3, j], 3, j, "BH-to-stellar mass on the clock")
finish_region_fig(fig, f"ks_regions_agn_{FIDUCIAL_AP}.png",
                  "SIMBA-25 quenched galaxies by KS region — AGN coupling class, coupling strength, Eddington ratio and M_BH/M* on the quench clock")
AGN_FRAC = Table(rows=AGN_FRAC)
for (zlo, zhi, _), ttl in zip(Z_COLS, ZCOL_NAME):
    _r = AGN_FRAC[(AGN_FRAC["z_lo"] == zlo) & (AGN_FRAC["z_hi"] == zhi)]
    print(f"{ttl:8s} " + " | ".join(f"{reg[:6]} " + " ".join(f"{c[:3]} {100 * float(_r[(_r['region'] == reg) & (_r['agn_class'] == c)]['fraction'][0]):3.0f}%"
                                                                    for c in CLASS_ORDER) for reg in REGION_GROUPS))

In [ ]:
# ── Figure 6 — the dust-fraction / stellar-age plane along the clock ──
# (a) the plane: per region the median history track (age_mw, log M_dust/M*) resampled on the clock (square = QT, small
#     dots every 0.5 Gyr, labels at -1 / +1 / +2 Gyr), the stage medians (SFT triangle, QT square, end circle), every galaxy at
#     its track end (small dots) and the SF controls at their anchor (grey); (b) log M_dust/M* on the clock; (c) mass-weighted
#     stellar age on the clock (caesar `ages.mass_weighted`: the mean age of the stars, so it grows ~1:1 with time once
#     nothing forms — the differences between the groups are the signal, not the slope).
_md, _ms = np.asarray(HIST["mdust"], float), np.asarray(HIST["mstar"], float)
with np.errstate(divide="ignore", invalid="ignore"):
    LOG_FDUST = np.where((_md > 0) & (_ms > 0), np.log10(_md / _ms), np.nan)
AGE_MW = np.asarray(HIST["age_mw"], float)
_hp = _s(HIST["pop"])
_haz = np.asarray(HIST["anchor_z"], float)
_TICK_DT = np.arange(DT_GRID[0], DT_GRID[-1] + 1e-6, 0.5)
_LABEL_DT = (-1.0, 1.0, 2.0)
_end_age, _end_fd = hist_at_stage("end", AGE_MW), hist_at_stage("end", LOG_FDUST)

fig, axs = region_grid(3, height=3.8)
PLANE_ROWS = []
for j, (zlo, zhi, ttl) in enumerate(Z_COLS):
    ax = axs[0, j]
    msf = (_hp == "SF") & (_haz > zlo) & (_haz <= zhi)
    ax.scatter(AGE_MW[msf], LOG_FDUST[msf], s=7, c=C_SF, alpha=0.35, lw=0, zorder=1, label=f"SF controls at the anchor (N={int(msf.sum())})")
    for reg in REGION_GROUPS:
        m_all = members(zlo, zhi, reg)
        m = m_all & np.asarray(GALS["has_clock"], bool)
        gk = np.asarray(GALS["gkey"])[m]
        col = REGION_COLOR[reg]
        ax.scatter(_end_age[m_all], _end_fd[m_all], s=9, c=[col], alpha=0.45, lw=0, zorder=2)
        if len(gk):
            sa, sf_ = kl.grid_stats(grid_tracks(gk, AGE_MW), NMIN_GRID), kl.grid_stats(grid_tracks(gk, LOG_FDUST), NMIN_GRID)
            ok = np.isfinite(sa["med"]) & np.isfinite(sf_["med"])
            if ok.sum() >= 2:
                ax.plot(sa["med"][ok], sf_["med"][ok], color=col, lw=2.2, zorder=4, label=f"{REGION_SHORT[reg]} median track (N={len(gk)})")
                tk = ok & np.isin(np.round(DT_GRID, 3), np.round(_TICK_DT, 3))
                ax.scatter(sa["med"][tk], sf_["med"][tk], s=14, c=[col], zorder=5)
                for dt0 in _LABEL_DT:
                    k = int(np.argmin(np.abs(DT_GRID - dt0)))
                    if ok[k]:
                        ax.annotate(f"{dt0:+.0f}", (sa["med"][k], sf_["med"][k]), xytext=(3, 3), textcoords="offset points", fontsize=7, color=col)
                for k, dt0 in enumerate(DT_GRID):
                    PLANE_ROWS.append(dict(z_lo=zlo, z_hi=zhi, region=reg, dt_qt_gyr=dt0, n=int(min(sa["n"][k], sf_["n"][k])),
                                           age_mw_med=sa["med"][k], log_fdust_med=sf_["med"][k]))
        # stage medians (all galaxies of the region with that stage row)
        for st in STAGES_DRAW:
            a_st, f_st = hist_at_stage(st, AGE_MW)[m_all], hist_at_stage(st, LOG_FDUST)[m_all]
            okst = np.isfinite(a_st) & np.isfinite(f_st)
            if okst.sum() >= NMIN_ECDF:
                ax.scatter(np.median(a_st[okst]), np.median(f_st[okst]), s=95 if st in END_STAGES else 60, marker=STAGE_MARKER[st], c=[col],
                           edgecolors="k", linewidths=0.9, zorder=6)
    ax.set_xlabel("mass-weighted stellar age [Gyr]"); ax.set_ylabel(r"$\log(M_{\rm dust}/M_\star)$")
    ax.set_xlim(0, 9.5); ax.set_ylim(-6.2, -1.6); ax.grid(alpha=0.15, lw=0.6)
    ax.legend(fontsize=7, loc="lower left", frameon=False)
    letter(ax, 0, j, "dust fraction vs age")
    draw_clock(axs[1, j], zlo, zhi, LOG_FDUST, r"$\log(M_{\rm dust}/M_\star)$")
    letter(axs[1, j], 1, j, "dust fraction on the clock")
    draw_clock(axs[2, j], zlo, zhi, AGE_MW, "mass-weighted stellar age [Gyr]")
    letter(axs[2, j], 2, j, "stellar age on the clock")
_stage_items = [(Line2D([], [], marker=STAGE_MARKER[st], ls="", ms=8, mfc="0.5", mec="k"), f"median at {STAGE_LABEL[st]}") for st in STAGES_DRAW]
_stage_items += [(Line2D([], [], marker="s", ls="", ms=5, mfc="0.5", mec="none"), "track dots every 0.5 Gyr of the clock, labels = t − t_QT [Gyr]"),
                 (Line2D([], [], marker="o", ls="", ms=4, mfc="0.5", mec="none", alpha=0.6), "individual galaxies at the track end")]
fig.legend([h for h, _ in _stage_items], [l for _, l in _stage_items], loc="lower center", ncol=5, frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, -0.01))
finish_region_fig(fig, f"ks_regions_fdust_age_{FIDUCIAL_AP}.png",
                  "SIMBA-25 quenched galaxies by KS region — the dust-fraction / stellar-age plane on the quench clock")
PLANE_TRACKS = Table(rows=PLANE_ROWS)
PLANE_TRACKS.to_pandas().to_csv(os.path.join(KSDIR, "ks_region_fdust_age_tracks.csv"), index=False)
print(f"median (age, log fdust) tracks on the clock -> {os.path.join(KSDIR, 'ks_region_fdust_age_tracks.csv')}")
for reg in REGION_GROUPS:
    m = members(-1, 99, reg)
    a, f = _end_age[m], _end_fd[m]
    print(f"  {REGION_SHORT[reg]:14s} at the track end (all z): age {np.nanmedian(a):.2f} Gyr, log fdust {np.nanmedian(f):+.2f}  (N={int(np.isfinite(f).sum())})")

In [ ]:
# ── Figure 7 — rotation along the clock (caesar catalogue kinematics of the tracked progenitors) ──
# kappa_rot = fraction of kinetic energy in ordered rotation (Sales+12) of the CAESAR members: (a) stars, (b) member gas
# (NaN where the progenitor has < NGAS_MIN gas particles), (c) stellar B/T; (d) kappa_rot of the stars at the track end
# (ECDF, SF controls dashed); (e, when tables/annulus_kinematics.fits of the powderday notebook exists) kappa_rot of the
# H2-weighted gas inside 10 kpc at the ANCHOR (Stage-0 cutouts; only the galaxies it covers).
_ng = np.asarray(HIST["ngas"], float)
KAPPA_STAR = np.asarray(HIST["kappa_star"], float)
KAPPA_GAS = np.where(_ng >= NGAS_MIN, np.asarray(HIST["kappa_gas"], float), np.nan)
BT_STAR = np.asarray(HIST["bt_star"], float)
_end_ks, _end_kg = hist_at_stage("end", KAPPA_STAR), hist_at_stage("end", KAPPA_GAS)
_hp, _haz = _s(HIST["pop"]), np.asarray(HIST["anchor_z"], float)

KIN_FITS = os.path.join(TABLEDIR, "annulus_kinematics.fits")
KIN_AP = "ap10kpc"
_kh2 = None
if os.path.exists(KIN_FITS):
    _K = Table.read(KIN_FITS)
    _K = _K[_s(_K["aperture"]) == KIN_AP]
    _kd = {f"{a}_{g}": float(v) for a, g, v in zip(np.asarray(_K["snap"]), np.asarray(_K["gal_id"]), np.asarray(_K["kappa_H2"], float))}
    _kh2 = np.array([_kd.get(g, np.nan) for g in np.asarray(GALS["gkey"])])
    _sfk = np.array([_kd.get(f"{a}_{g}", np.nan) for a, g in zip(np.asarray(HIST["anchor_snap"])[_hp == "SF"], np.asarray(HIST["gal_id"])[_hp == "SF"])])
    print(f"anchor kappa_H2({KIN_AP}) from {KIN_FITS}: {int(np.isfinite(_kh2).sum())}/{len(_kh2)} quenched, {int(np.isfinite(_sfk).sum())} SF controls")
else:
    print(f"no {KIN_FITS} (powderday notebook Part 8j0) -> panel (e) skipped")

fig, axs = region_grid(4 + (_kh2 is not None), height=3.3)
for j, (zlo, zhi, ttl) in enumerate(Z_COLS):
    draw_clock(axs[0, j], zlo, zhi, KAPPA_STAR, r"$\kappa_{\rm rot}$ (stars)")
    letter(axs[0, j], 0, j, "stellar rotation on the clock")
    draw_clock(axs[1, j], zlo, zhi, KAPPA_GAS, rf"$\kappa_{{\rm rot}}$ (member gas, $n_{{\rm gas}}\geq{NGAS_MIN}$)")
    letter(axs[1, j], 1, j, "gas rotation on the clock")
    draw_clock(axs[2, j], zlo, zhi, BT_STAR, "stellar B/T")
    letter(axs[2, j], 2, j, "bulge fraction on the clock")
    msf = (_hp == "SF") & (_haz > zlo) & (_haz <= zhi)
    ecdf_panel(axs[3, j], zlo, zhi, _end_ks, r"$\kappa_{\rm rot}$ (stars) at the track end", sf_values=KAPPA_STAR[msf], xlim=(0, 1))
    letter(axs[3, j], 3, j, "stellar rotation at the end")
    if _kh2 is not None:
        ecdf_panel(axs[4, j], zlo, zhi, _kh2, rf"$\kappa_{{\rm rot}}$ (H$_2$-weighted gas, $r<10$ kpc) at the anchor", sf_values=_sfk[msf[_hp == "SF"]], xlim=(0, 1))
        letter(axs[4, j], 4, j, "H$_2$ rotation at the anchor (cutouts)")
for ax in axs[:3].ravel():
    ax.set_ylim(0, 1)
finish_region_fig(fig, f"ks_regions_rotation_{FIDUCIAL_AP}.png",
                  "SIMBA-25 quenched galaxies by KS region — ordered rotation of stars and gas, bulge fraction, on the quench clock")
for reg in REGION_GROUPS:
    m = members(-1, 99, reg)
    print(f"  {REGION_SHORT[reg]:14s} at the track end (all z): kappa_star {np.nanmedian(_end_ks[m]):.2f} (N={int(np.isfinite(_end_ks[m]).sum())}), "
          f"kappa_gas {np.nanmedian(_end_kg[m]):.2f} (N={int(np.isfinite(_end_kg[m]).sum())})"
          + (f", kappa_H2 anchor {np.nanmedian(_kh2[m]):.2f} (N={int(np.isfinite(_kh2[m]).sum())})" if _kh2 is not None else ""))

In [ ]:
# ── Figure 8 — molecular content and extent along the clock ──
# (a) M_H2/M* (catalogue, x1.36 He; no H2 -> NaN) on the clock; (b) caesar gas half-mass radius (all member gas, 3D, proper
# kpc; 0 = no gas -> NaN) on the clock; (c) its ratio to the stellar half-mass radius (gas more/less extended than the stars);
# (d) the face-on member H2 half-mass radius R50(H2) of Part 3 at the drawn stages SFT -> QT -> end (median, 16-84 % bars,
# N); (e) R50(H2)/R50(*) at the same stages (both face-on member radii).
_mh2, _ms, _rg, _rs = (np.asarray(HIST[c], float) for c in ("mh2", "mstar", "r50gas", "r50star"))
with np.errstate(divide="ignore", invalid="ignore"):
    LOG_FH2 = np.where((_mh2 > 0) & (_ms > 0), np.log10(HE_FACTOR * _mh2 / _ms), np.nan)
    R50GAS = np.where(_rg > 0, _rg, np.nan)
    R50GAS_STAR = np.where((_rg > 0) & (_rs > 0), _rg / _rs, np.nan)
_xoff = {reg: (k - 1) * 0.22 for k, reg in enumerate(REGION_GROUPS)}


def stage_strip(ax, zlo, zhi, per_stage, ylabel, log=False):
    """Median with 16-84 % bars of the per-galaxy arrays `per_stage[stage]` (GALS order) per region at every drawn stage;
    x = stage (regions offset). Returns {(region, stage): (n, med, p16, p84)}."""
    out = {}
    for reg in REGION_GROUPS:
        m = members(zlo, zhi, reg)
        xs, ys, lo, hi = [], [], [], []
        for k, st in enumerate(STAGES_DRAW):
            v = np.asarray(per_stage[st], float)[m]
            v = v[np.isfinite(v)]
            out[(reg, st)] = (len(v), np.median(v) if len(v) else np.nan, np.percentile(v, 16) if len(v) else np.nan, np.percentile(v, 84) if len(v) else np.nan)
            if len(v) >= NMIN_ECDF:
                xs.append(k + _xoff[reg]); ys.append(np.median(v)); lo.append(np.median(v) - np.percentile(v, 16)); hi.append(np.percentile(v, 84) - np.median(v))
                ax.text(k + _xoff[reg], np.percentile(v, 84), str(len(v)), fontsize=6.5, color=REGION_COLOR[reg], ha="center", va="bottom")
        if xs:
            ax.errorbar(xs, ys, yerr=[lo, hi], fmt="-", color=REGION_COLOR[reg], lw=1.6, capsize=3, zorder=4, label=REGION_SHORT[reg])
            for x0, y0, st in zip(xs, ys, [s for s in STAGES_DRAW if (reg, s) in out and out[(reg, s)][0] >= NMIN_ECDF]):
                ax.scatter(x0, y0, s=60 if st in END_STAGES else 40, marker=STAGE_MARKER[st], c=[REGION_COLOR[reg]],
                           edgecolors="k" if st in END_STAGES else "none", linewidths=0.8, zorder=5)
    ax.set_xticks(range(len(STAGES_DRAW))); ax.set_xticklabels([STAGE_LABEL[s] for s in STAGES_DRAW], fontsize=8)
    ax.set_xlim(-0.6, len(STAGES_DRAW) - 0.4); ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale("log")
    ax.grid(alpha=0.15, lw=0.6); ax.legend(fontsize=7.5, frameon=False)
    return out


R50H2_ST = {st: np.asarray(GALS[f"r50h2_{st}"], float) for st in STAGES_DRAW}
with np.errstate(divide="ignore", invalid="ignore"):
    R50H2_STAR_ST = {st: np.asarray(GALS[f"r50h2_{st}"], float) / np.asarray(GALS[f"r50star_{st}"], float) for st in STAGES_DRAW}

fig, axs = region_grid(5, height=3.3)
EXTENT_ROWS = []
for j, (zlo, zhi, ttl) in enumerate(Z_COLS):
    draw_clock(axs[0, j], zlo, zhi, LOG_FH2, r"$\log(M_{\rm H_2}/M_\star)$")
    letter(axs[0, j], 0, j, r"molecular fraction ($\times1.36$ He) on the clock")
    draw_clock(axs[1, j], zlo, zhi, R50GAS, r"$R_{50}$(gas) [kpc]")
    letter(axs[1, j], 1, j, "gas extent (catalogue, 3D, proper) on the clock")
    draw_clock(axs[2, j], zlo, zhi, R50GAS_STAR, r"$R_{50}({\rm gas}) / R_{50}(\star)$")
    letter(axs[2, j], 2, j, "gas vs stellar extent (catalogue) on the clock")
    for i, (per, lab, key) in enumerate(((R50H2_ST, r"$R_{50}({\rm H_2})$ [kpc]", "r50_H2"),
                                         (R50H2_STAR_ST, r"$R_{50}({\rm H_2}) / R_{50}(\star)$", "r50_H2_over_star")), start=3):
        res = stage_strip(axs[i, j], zlo, zhi, per, lab, log=True)
        letter(axs[i, j], i, j, "H$_2$ extent (face-on members) at the stages" if i == 3 else "H$_2$ vs stellar extent (face-on members) at the stages")
        for (reg, st), (n, med, p16, p84) in res.items():
            EXTENT_ROWS.append(dict(quantity=key, z_lo=zlo, z_hi=zhi, region=reg, stage=st, n=n, med=med, p16=p16, p84=p84))
for ax in axs[2]:
    ax.axhline(1, color="0.4", lw=0.8, ls="-.")
for ax in axs[4]:
    ax.axhline(1, color="0.4", lw=0.8, ls="-.")
finish_region_fig(fig, f"ks_regions_h2_extent_{FIDUCIAL_AP}.png",
                  "SIMBA-25 quenched galaxies by KS region — molecular fraction, gas extent on the clock, H$_2$ extent at the drawn stages")
EXTENT = Table(rows=EXTENT_ROWS)
EXTENT.to_pandas().to_csv(os.path.join(KSDIR, "ks_region_h2_extent.csv"), index=False)
print(f"H2 extent at the stages -> {os.path.join(KSDIR, 'ks_region_h2_extent.csv')}")
_e = EXTENT[(EXTENT["z_lo"] == Z_ALL[0][0]) & (EXTENT["quantity"] == "r50_H2")]
for reg in REGION_GROUPS:
    print(f"  {REGION_SHORT[reg]:14s} R50(H2) [kpc] all z: " + "  ".join(
        f"{st} {float(r['med']):.2f} (N={int(r['n'])})" for st in STAGES_DRAW for r in _e[(_e["region"] == reg) & (_e["stage"] == st)]))

In [ ]:
# ── Part 5 summary — one row per (z column, region): sizes, composition, medians at the track end, KS p-values below vs above ──
_end_vals = {"age_mw_end": hist_at_stage("end", AGE_MW), "log_fdust_end": hist_at_stage("end", LOG_FDUST),
             "kappa_star_end": hist_at_stage("end", KAPPA_STAR), "kappa_gas_end": hist_at_stage("end", KAPPA_GAS),
             "bt_star_end": hist_at_stage("end", BT_STAR), "log_fedd_end": hist_at_stage("end", LOG_FEDD),
             "log_fh2_end": hist_at_stage("end", LOG_FH2), "r50gas_end_kpc": hist_at_stage("end", R50GAS),
             "r50_H2_end_kpc": R50H2_ST["end"], "r50_H2_over_star_end": R50H2_STAR_ST["end"]}
_clock_vals = {"z_qt": np.asarray(GALS["z_qt"], float), "log_tau_q": TIMING_ROWS[1][1], "t_decline_gyr": TIMING_ROWS[2][1],
               "dt_end_gyr": np.asarray(GALS["dt_end_gyr"], float), "xstr_quench": np.asarray(GALS["xstr_quench"], float)}
_cls = np.asarray(GALS["agn_class"]).astype(str)
_rows = []
for zlo, zhi, ttl in Z_COLS:
    samp = {}
    for reg in REGION_GROUPS + ["undef"]:
        m = members(zlo, zhi, reg)
        r = dict(z_lo=zlo, z_hi=zhi, region=reg, n=int(m.sum()), n_no_event=int((m & ~np.asarray(GALS["has_clock"], bool)).sum()),
                 log_mstar_med=np.nanmedian(np.asarray(GALS["log_mstar_anchor"], float)[m]) if m.any() else np.nan)
        for c in CLASS_ORDER:
            r[f"f_{c}"] = float((m & (_cls == c)).sum()) / m.sum() if m.any() else np.nan
        for k, v in {**_clock_vals, **_end_vals}.items():
            vv = v[m]
            vv = vv[np.isfinite(vv)]
            r[f"{k}_med"] = float(np.median(vv)) if len(vv) else np.nan
            r[f"{k}_n"] = int(len(vv))
            samp[(reg, k)] = vv
        _rows.append(r)
    for r in _rows[-4:]:
        for k in list(_clock_vals) + list(_end_vals):
            r[f"{k}_p_below_above"] = ks_p(samp.get(("below", k)), samp.get(("above", k)))
            r[f"{k}_p_below_within"] = ks_p(samp.get(("below", k)), samp.get(("within", k)))
REGION_PROPS = Table(rows=_rows)
REGION_PROPS.to_pandas().to_csv(os.path.join(KSDIR, "ks_region_properties.csv"), index=False)
print(f"region properties -> {os.path.join(KSDIR, 'ks_region_properties.csv')}\n")
_show = ["z_qt", "log_tau_q", "dt_end_gyr", "xstr_quench", "age_mw_end", "log_fdust_end", "kappa_star_end", "kappa_gas_end", "log_fh2_end", "r50_H2_end_kpc"]
for (zlo, zhi, ttl), name in zip(Z_COLS, ZCOL_NAME):
    print(f"{name}  ({ttl.replace('$', '')})")
    print(f"  {'region':10s} {'N':>4s} {'noev':>4s} {'f_str':>5s} " + " ".join(f"{k[:12]:>12s}" for k in _show))
    for reg in REGION_GROUPS + ["undef"]:
        r = REGION_PROPS[(REGION_PROPS["z_lo"] == zlo) & (REGION_PROPS["z_hi"] == zhi) & (REGION_PROPS["region"] == reg)][0]
        print(f"  {reg:10s} {int(r['n']):4d} {int(r['n_no_event']):4d} {float(r['f_strong']):5.2f} " + " ".join(f"{float(r[k + '_med']):12.2f}" for k in _show))
    r = REGION_PROPS[(REGION_PROPS["z_lo"] == zlo) & (REGION_PROPS["z_hi"] == zhi) & (REGION_PROPS["region"] == "below")][0]
    print(f"  {'p(b vs a)':10s} {'':4s} {'':4s} {'':5s} " + " ".join(f"{float(r[k + '_p_below_above']):12.2g}" for k in _show))

In [ ]:
# ── Part 5c — presentation figures: no titles, no grid, first column = all anchors (thin frame), then the three z ranges ──
# Three figures, minimal text, legends carry only the number of galaxies in the group:
#   ks_pres_agn_<ap>        AGN class composition (sequential purples, distinct from the region colours) + coupling strength x_str
#   ks_pres_dust_<ap>       the dust-fraction / age plane, log M_dust/M* on the clock, dust-to-gas ratio on the clock + its ECDF at the end
#   ks_pres_kinematics_<ap> kappa_rot of the member gas and stellar B/T on the clock, each with its ECDF at the track end
from matplotlib.patches import Rectangle
PRES_COLS = [Z_ALL[0]] + Z_PANELS
PRES_NAME = [r"all anchors, $0.3\leq z\leq2$", r"$z\leq0.5$", r"$0.7\leq z\leq1$", r"$1.15\leq z\leq2$"]
CLASS_COLOR_PRES = {"weak": "#bcbddc", "intermediate": "#807dba", "strong": "#3f007d", "no_event": "0.85"}
CLASS_TEXT_PRES = {"weak": "0.15", "intermediate": "white", "strong": "white", "no_event": "0.15"}
CLASS_NAME = {"weak": "weak", "intermediate": "intermediate", "strong": "strong", "no_event": "no event"}
PRES_DPI = 200
_mdg, _mgg = np.asarray(HIST["mdust"], float), np.asarray(HIST["mgas"], float)
with np.errstate(divide="ignore", invalid="ignore"):
    LOG_DGR = np.where((_mdg > 0) & (_mgg > 0), np.log10(_mdg / _mgg), np.nan)      # dust-to-gas ratio (all member gas)
_hp, _haz = _s(HIST["pop"]), np.asarray(HIST["anchor_z"], float)


def _sf_mask(zlo, zhi):
    return (_hp == "SF") & (_haz > zlo) & (_haz <= zhi)


def pres_grid(nrows, height=3.1):
    fig, axs = plt.subplots(nrows, len(PRES_COLS), figsize=(4.4 * len(PRES_COLS), height * nrows), squeeze=False)
    for ax in axs.ravel():
        ax.grid(False)
        ax.tick_params(direction="in", top=True, right=True)
    for j, name in enumerate(PRES_NAME):
        axs[0, j].text(0.0, 1.02, name, transform=axs[0, j].transAxes, ha="left", va="bottom", fontsize=9.5, color="0.25")
    return fig, axs


def pres_finish(fig, axs, fname, pad=0.008):
    """Common axis limits along every row (columns comparable), thin frame around the first (all-anchors) column; save PNG + PDF."""
    for row in axs:
        row_ = [ax for ax in row if ax.has_data()]
        if len(row_) <= 1:
            continue
        if all(getattr(a, "_auto_y", False) for a in row_):      # clock rows without a fixed range: span of the drawn LINES (the
            ys = np.concatenate([np.asarray(l.get_ydata(), float) for a in row_ for l in a.get_lines()
                                 if len(l.get_ydata()) > 1 and l.get_transform() == a.transData])   # not the axvline (axes-fraction y)
            ys = ys[np.isfinite(ys)]                              # 16-84 bands may run off to dust-free outliers and are clipped)
            if ys.size:
                lo, hi = ys.min(), ys.max()
                for a in row_:
                    a.set_ylim(lo - 0.15 * (hi - lo), hi + 0.15 * (hi - lo))
        else:
            lims = np.array([a.get_ylim() for a in row_])
            lo, hi = (lims.min(), lims.max()) if lims[0, 0] <= lims[0, 1] else (lims.max(), lims.min())   # inverted axes keep their sense
            for a in row_:
                a.set_ylim(lo, hi)
        lims = np.array([a.get_xlim() for a in row_])
        for a in row_:
            a.set_xlim(lims.min(), lims.max())
    plt.tight_layout(w_pad=1.2, h_pad=1.0)
    fig.canvas.draw()
    inv = fig.transFigure.inverted()
    bbs = [ax.get_tightbbox(fig.canvas.get_renderer()).transformed(inv) for ax in axs[:, 0]]
    x0, x1 = min(b.x0 for b in bbs) - pad, max(b.x1 for b in bbs) + pad
    y0, y1 = min(b.y0 for b in bbs) - pad, max(b.y1 for b in bbs) + pad * 1.6
    fig.add_artist(Rectangle((x0, y0), x1 - x0, y1 - y0, transform=fig.transFigure, fill=False, ec="0.4", lw=0.9, zorder=20))
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIGDIR, f"{fname}.{ext}"), dpi=PRES_DPI, bbox_inches="tight")
    plt.show()


def _legend(ax, loc="best", **kw):
    kw.setdefault("handlelength", 1.8)
    ax.legend(frameon=False, fontsize=8, loc=loc, **kw)


def pres_clock(ax, zlo, zhi, col, ylabel, ylim=None, legend=True):
    n_sf = 0
    for reg in REGION_GROUPS:
        gk = np.asarray(GALS["gkey"])[members(zlo, zhi, reg, need_clock=True)]
        if not len(gk):
            continue
        st = kl.grid_stats(grid_tracks(gk, col), NMIN_GRID)
        ok = np.isfinite(st["med"])
        if not ok.any():
            continue
        ax.plot(DT_GRID[ok], st["med"][ok], color=REGION_COLOR[reg], lw=2.2, zorder=4, label=f"{REGION_SHORT[reg]} (N={len(gk)})")
        ax.fill_between(DT_GRID[ok], st["p16"][ok], st["p84"][ok], color=REGION_COLOR[reg], alpha=0.12, lw=0, zorder=2)
    v = sf_band(ax, zlo, zhi, col)
    if len(v) >= NMIN_ECDF:
        ax.plot([], [], color=C_SF, lw=1.0, ls="--", label=f"SF controls (N={len(v)})")
    ax.axvline(0, color="0.5", lw=0.8, ls=":")
    ax.set_xlim(DT_GRID[0], DT_GRID[-1]); ax.set_xlabel(r"$t - t_{\rm QT}$ [Gyr]"); ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax._auto_y = ylim is None
    if legend:
        _legend(ax)


def pres_ecdf(ax, zlo, zhi, values, xlabel, sf_values=None, xlim=None, legend=True):
    vals = np.asarray(values, float)
    for reg in REGION_GROUPS:
        m = members(zlo, zhi, reg) & np.isfinite(vals)
        if m.sum() < NMIN_ECDF:
            continue
        x, F = kl.ecdf(vals[m])
        ax.step(np.r_[x[0], x], np.r_[0, F], where="post", color=REGION_COLOR[reg], lw=2.0, zorder=4, label=f"{REGION_SHORT[reg]} (N={int(m.sum())})")
        ax.plot([np.median(vals[m])], [0.5], marker="|", ms=12, mew=2.0, color=REGION_COLOR[reg], zorder=5)
    if sf_values is not None:
        sv = np.asarray(sf_values, float)
        sv = sv[np.isfinite(sv)]
        if len(sv) >= NMIN_ECDF:
            x, F = kl.ecdf(sv)
            ax.step(np.r_[x[0], x], np.r_[0, F], where="post", color=C_SF, lw=1.2, ls="--", zorder=3, label=f"SF controls (N={len(sv)})")
    ax.set_ylim(0, 1.02); ax.set_ylabel("cumulative fraction"); ax.set_xlabel(xlabel)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if legend:
        _legend(ax, loc="lower right")


_cls = np.asarray(GALS["agn_class"]).astype(str)

# ── figure A: AGN class composition + coupling strength ──
fig, axs = pres_grid(2, height=3.1)
for j, (zlo, zhi, _) in enumerate(PRES_COLS):
    ax = axs[0, j]
    for k, reg in enumerate(REGION_GROUPS):
        m = members(zlo, zhi, reg)
        n, left = int(m.sum()), 0.0
        for c in CLASS_ORDER:
            f = float((m & (_cls == c)).sum()) / n if n else 0.0
            ax.barh(k, f, left=left, color=CLASS_COLOR_PRES[c], edgecolor="white", lw=0.6, height=0.62, label=CLASS_NAME[c] if k == 0 else None)
            if f >= 0.08:
                ax.text(left + f / 2, k, f"{100 * f:.0f}%", ha="center", va="center", fontsize=7.5, color=CLASS_TEXT_PRES[c])
            left += f
        ax.text(1.02, k, f"N={n}", va="center", fontsize=8)
    ax.set_yticks(range(len(REGION_GROUPS))); ax.set_yticklabels([REGION_SHORT[r] for r in REGION_GROUPS], fontsize=8.5)
    for k, r in enumerate(REGION_GROUPS):
        ax.get_yticklabels()[k].set_color(REGION_COLOR[r])
    ax.set_xlim(0, 1.2); ax.set_ylim(3.35, -0.55); ax.set_xticks(np.linspace(0, 1, 6)); ax.set_xlabel("fraction of the group")
    ax.tick_params(axis="y", length=0)
    if j == 0:
        _legend(ax, loc="lower left", ncol=4, columnspacing=1.0, handlelength=1.2)
    pres_ecdf(axs[1, j], zlo, zhi, np.asarray(GALS["xstr_quench"], float), r"$x_{\rm str}$ (AGN coupling over [SFT, QT])", legend=(j == 0))
pres_finish(fig, axs, f"ks_pres_agn_{FIDUCIAL_AP}")

# ── figure B: dust fraction vs age, dust fraction and dust-to-gas ratio on the clock, DGR at the end ──
_end_age, _end_fd, _end_dgr = hist_at_stage("end", AGE_MW), hist_at_stage("end", LOG_FDUST), hist_at_stage("end", LOG_DGR)
_TICK = np.isin(np.round(DT_GRID, 3), np.round(np.arange(DT_GRID[0], DT_GRID[-1] + 1e-6, 0.5), 3))
fig, axs = pres_grid(4, height=3.2)
for j, (zlo, zhi, _) in enumerate(PRES_COLS):
    ax = axs[0, j]
    msf = _sf_mask(zlo, zhi)
    ax.scatter(AGE_MW[msf], LOG_FDUST[msf], s=7, c=C_SF, alpha=0.35, lw=0, zorder=1, label=f"SF controls (N={int(msf.sum())})")
    for reg in REGION_GROUPS:
        m_all = members(zlo, zhi, reg)
        gk = np.asarray(GALS["gkey"])[m_all & np.asarray(GALS["has_clock"], bool)]
        col = REGION_COLOR[reg]
        ax.scatter(_end_age[m_all], _end_fd[m_all], s=9, c=[col], alpha=0.45, lw=0, zorder=2)
        if len(gk):
            sa, sf_ = kl.grid_stats(grid_tracks(gk, AGE_MW), NMIN_GRID), kl.grid_stats(grid_tracks(gk, LOG_FDUST), NMIN_GRID)
            ok = np.isfinite(sa["med"]) & np.isfinite(sf_["med"])
            if ok.sum() >= 2:
                ax.plot(sa["med"][ok], sf_["med"][ok], color=col, lw=2.2, zorder=4, label=f"{REGION_SHORT[reg]} (N={int(m_all.sum())})")
                ax.scatter(sa["med"][ok & _TICK], sf_["med"][ok & _TICK], s=12, c=[col], zorder=5)
        for st in STAGES_DRAW:
            a_st, f_st = hist_at_stage(st, AGE_MW)[m_all], hist_at_stage(st, LOG_FDUST)[m_all]
            okst = np.isfinite(a_st) & np.isfinite(f_st)
            if okst.sum() >= NMIN_ECDF:
                ax.scatter(np.median(a_st[okst]), np.median(f_st[okst]), s=90 if st in END_STAGES else 58, marker=STAGE_MARKER[st], c=[col],
                           edgecolors="k", linewidths=0.9, zorder=6)
    ax.set_xlabel("mass-weighted stellar age [Gyr]"); ax.set_ylabel(r"$\log(M_{\rm dust}/M_\star)$")
    ax.set_xlim(0, 9.5); ax.set_ylim(-6.2, -1.6)
    if j == 0:
        _legend(ax, loc="lower left")
    pres_clock(axs[1, j], zlo, zhi, LOG_FDUST, r"$\log(M_{\rm dust}/M_\star)$", legend=(j == 0))
    pres_clock(axs[2, j], zlo, zhi, LOG_DGR, r"$\log(M_{\rm dust}/M_{\rm gas})$", legend=(j == 0))
    pres_ecdf(axs[3, j], zlo, zhi, _end_dgr, r"$\log(M_{\rm dust}/M_{\rm gas})$ at the track end", sf_values=LOG_DGR[msf], legend=(j == 0))
_items = [(Line2D([], [], marker=STAGE_MARKER[st], ls="", ms=8, mfc="0.55", mec="k"), STAGE_LABEL[st]) for st in STAGES_DRAW]
_items.append((Line2D([], [], marker="o", ls="", ms=4, mfc="0.55", mec="none"), "median track, dots every 0.5 Gyr"))
axs[0, 1].legend([h for h, _ in _items], [l for _, l in _items], frameon=False, fontsize=8, loc="lower left", handlelength=1.8)
pres_finish(fig, axs, f"ks_pres_dust_{FIDUCIAL_AP}")

# ── figure C: gas rotation and stellar bulge fraction on the clock, each with its ECDF at the track end ──
_end_kg, _end_bt = hist_at_stage("end", KAPPA_GAS), hist_at_stage("end", BT_STAR)
fig, axs = pres_grid(4, height=3.1)
for j, (zlo, zhi, _) in enumerate(PRES_COLS):
    msf = _sf_mask(zlo, zhi)
    pres_clock(axs[0, j], zlo, zhi, KAPPA_GAS, r"$\kappa_{\rm rot}$ (gas)", ylim=(0, 1), legend=(j == 0))
    pres_ecdf(axs[1, j], zlo, zhi, _end_kg, r"$\kappa_{\rm rot}$ (gas) at the track end", sf_values=KAPPA_GAS[msf], xlim=(0, 1), legend=(j == 0))
    pres_clock(axs[2, j], zlo, zhi, BT_STAR, "B/T (stars)", ylim=(0, 1), legend=(j == 0))
    pres_ecdf(axs[3, j], zlo, zhi, _end_bt, "B/T (stars) at the track end", sf_values=BT_STAR[msf], xlim=(0, 1), legend=(j == 0))
pres_finish(fig, axs, f"ks_pres_kinematics_{FIDUCIAL_AP}")
print("presentation figures ->", ", ".join(f"ks_pres_{k}_{FIDUCIAL_AP}.png/.pdf" for k in ("agn", "dust", "kinematics")), "in", FIGDIR)

## Conventions, caveats, and what the pieces are

* **Particles**: only CAESAR members (`gal.glist` / `gal.slist`, the `member` flag of the reduced
  files); the 100 kpc aperture of the files (CGM, satellites) is ignored. Face-on = the stellar
  principal frame stored in each file (`pos @ evecs`, columns = axes — the older
  `quench_mode_vs_sigma_gas` / `box_resolution_comparison` helper uses `evecs.T`, which is the wrong
  frame). For spheroidal quenched galaxies the frame is nearly degenerate and "face-on" is only nominal.
* **Σ$_{\rm H_2}$**: SIMBA's per-particle H$_2$ (`m_H2 = m_gas X_H f_neut f_mol`) is hydrogen-only; the
  observed M$_{\rm H_2}$ (α$_{\rm CO}$ = 4.36) includes helium → ×1.36 on the simulated values in the
  plots and summaries; raw masses stay in the tables.
* **0.5 factor**: inside a half-mass radius M(<R$_{50}$)/πR$_{50}^2$ ≡ 0.5 M$_{\rm tot}$/πR$_{50}^2$, so the
  fiducial rows reproduce the observed convention with no extra factor. For the SFR the observed
  points assume the SFR follows the CO (0.5 SFR$_{\rm tot}$/πR$_{\rm CO}^2$): that is `logSigmaSFR_obs`
  (used on the R50_H2 rows); `logSigmaSFR_inside` is the literal SFR(<R)/area. Fixed apertures
  (1 / 3.16 / 10 kpc) never carry a 0.5 and are **not plotted**: a fixed aperture is a different Σ
  definition from the observed one — for a compact quenched galaxy both Σ are diluted by (R$_{50}$/r)$^2$
  and the point slides along a slope-1 line, so its offset from a slope-1.4 relation and its distance from
  the observed points are aperture artefacts. They stay in the measurement table and in `ks_stage_summary`
  (comparable to the `cluster_tracks.py` overlay of the pilot_specphot project).
* **SFR**: fiducial = mass of member stars formed within 100 Myr of the snapshot / 100 Myr (current
  masses, no mass-loss correction, ~5–10 %); `sfr25` and the instantaneous gas SFR (`sfr_inst`, exactly
  0 when no gas sits above the SF density threshold) are stored. SFR = 0 → censored: the one-particle
  floor m$_\star$/100 Myr/area is drawn as a downward arrow and excluded from the medians.
* **Stages**: `sf_peak` = peak of the (median-3 smoothed) sSFR history; `sft`/`qt` = the last quench
  event of `find_quenching_times` (1/t and 0.2/t crossings with persistence), identical to the selection
  table; `post_quench` = end of the persistence window (QT + 0.2 QT) — often beyond the anchor, then
  not measured; `gas_min` = trough of the H$_2$ mass after QT (before any floor); `anchor` = the
  selection snapshot; `end` = the drawn track endpoint: the anchor row when the catalogue M$_{\rm H_2}$/M$_\star$
  at the anchor exceeds `FH2_MIN_END` = 10$^{-4}$, otherwise the **last** history snapshot above that fraction
  (`end_is_anchor`, `fh2_anchor` in the epochs table; no end row if the history never exceeds it). Galaxies
  without a detected event (28/266) have sf_peak / gas_min / anchor / end only.
* **Floors**: Σ from gas needs ≥ `NGAS_MIN` = 10 gas particles in the aperture, R$_{50}$(H$_2$) needs
  ≥ `NH2_MIN` = 5 H$_2$-bearing particles (else the fiducial row is NaN — the summary reports the
  fraction); cis25 particle masses are ~1.6×10$^6$ M$_\odot$ (stars) so the 100 Myr SFR floor is
  ~0.016 M$_\odot$ yr$^{-1}$.
* **Binned tracks** (Part 4c): R$_{50}$(H$_2$) rows only, stages `STAGES_DRAW` = SFT → QT → end — nothing else is
  required of a galaxy (sf_peak, post-quench and the H$_2$ trough are measured and summarised but not drawn on
  the plane; post-quench = QT + persistence lies beyond the anchor for 57 % of the sample), per-stage medians
  joined in time order, no scatter bars. Every bin property is fixed at the galaxy's **anchor** — catalogue
  SFR (instantaneous; never exactly 0 in the catalogue, unlike the 100 Myr archaeological value),
  `log_mstar` of the selection, M$_{\rm dust}$/M$_\star$ (catalogue masses), Σ$_{\rm dust}$ = 0.5 M$_{\rm dust}$/πR$_{50\star}^2$
  with the face-on member stellar half-mass radius (catalogue 3D radius as fallback), the galaxy's own
  Σ$_{\rm H_2}$ (`logSigmaH2` of its R$_{50}$(H$_2$) `end` row, ×1.36 He — i.e. the plane's x-coordinate at
  the track end: it sorts the endpoints left→right by construction and is undefined where R$_{50}$(H$_2$) is,
  so the question it answers is whether galaxies that *end* at high Σ$_{\rm H_2}$ travelled differently),
  the molecular fraction M$_{\rm H_2}$/M$_\star$ (catalogue H$_2$ ×1.36 He, the observed μ$_{\rm H_2}$ scale), the
  **region of the track end w.r.t. the reference relation** `KS_REF` = B08 (`ks_region_anchor`: Δ = `logSigmaSFR` − B08(`logSigmaH2`) on the R$_{50}$(H$_2$)
  `end` row; below / within / above the ±0.20 dex published-scatter band; a censored anchor is "below" only when
  its one-particle floor already lies below the band, otherwise undetermined and not binned — the endpoints are
  sorted by construction, so the tracks answer where the galaxies of each region *come from*; Figure 0 applies the
  same rule, `ks_region`, at the anchor (0a) and at every critical point itself (0b by z range, 0c all z) and tabulates
  the counts in `ks_stage_regions.csv`), and `agn_class` of the selection table (`no_event` galaxies are left out of that scheme). Tercile edges come
  from the whole quenched sample (`EDGES_PER_PANEL=False`), so a bin means the same thing in the three z
  panels and the legend N shows how the bin populates each panel. A stage median needs ≥ `NMIN_BIN` = 4
  uncensored galaxies; with fewer (but ≥ 4 galaxies in total — typically the anchor of a bin full of SFR = 0
  galaxies, e.g. the "below K98" region) the median over all galaxies with the censored SFRs at their one-particle
  floor is drawn instead, as an open marker with a down arrow: floors are ≥ the true values, so that point is an
  *upper limit* on the stage median (`x_all`, `y_all`, `n_all` in `ks_binned_tracks.csv`). The dashed black track
  is the median of all quenched galaxies of the panel.
* **KS regions followed in time** (Part 5): the three `ks_region_anchor` groups (the `ksreg` bins) are the units; every figure has
  the three anchor-z columns plus an all-anchors column. `ks_track_histories.fits` (Part 5a, `OVERWRITE_HISTORIES`) holds one row
  per (anchor, galaxy, history snapshot): the history-file properties (masses, SFR, caesar mass-weighted stellar age,
  half-mass radii — the catalogue/history radii are **comoving** kpc and are stored there in proper kpc), the BH history
  (M$_{\rm BH}$, f$_{\rm Edd}$, Ṁ) and the caesar catalogue's `rotation.*` (κ$_{\rm rot}$, B/T of stars and member gas), central flag,
  100 Myr SFR and stellar metallicity of the tracked progenitor (plain h5py reads, one per snapshot); the SF controls have a
  single anchor row each (not in the history files → no clock). The clock is $t - t_{\rm QT}$: every history is linearly
  resampled onto `DT_GRID` (−2 … +3 Gyr, 0.25 steps; `ks_tracks_lib.interp_track`, no extrapolation) and the group median
  (16–84 %) is drawn where ≥ `NMIN_GRID` = 5 galaxies cover the point (`grid_stats`); coverage at late times is the early
  quenchers only. `no_event` galaxies (no t$_{\rm QT}$) enter only the clock-free panels (class composition, end-point ECDFs).
  κ$_{\rm rot}$ is caesar's (Sales+12 energy fraction over the members; gas κ masked below `NGAS_MIN` particles) — an
  H$_2$-weighted κ along the track would need particle velocities in the reduced files (not stored; the anchor-only H$_2$ κ of the
  powderday cutouts is shown when `tables/annulus_kinematics.fits` exists). ECDF panels print the two-sample KS p-value
  below-vs-above (scipy; NaN without it). Products: `ks_region_properties.csv`, `ks_region_fdust_age_tracks.csv`,
  `ks_region_h2_extent.csv`; figures `ks_regions_{timing,agn,fdust_age,rotation,h2_extent}_R50_H2.png`.
* **Cosmology**: Planck15 throughout (histories, quench finder, formation times) — SIMBA's own
  h = 0.68, Ω$_m$ = 0.30 differ at the sub-percent level in ages.
* **Observed side**: `obs_data/almac11/ks_table.csv` (see its README) — α$_{\rm CO}$ 4.36, R$_{31}$ 0.5,
  Σ = 0.5 M/πR$^2$ with R = CO(3–2) uv FWHM$_{\rm maj}$/2; unresolved sizes → Σ lower limits (slope-1
  arrows). Reference relation on the plane (`RELATIONS_DRAW`): **Bigiel+08** (Σ$_{\rm H_2}$ including He, 750 pc
  resolution, slope 0.96, its 0.20 dex scatter as the band) — an H$_2$ relation for H$_2$ surface densities, which is
  what both the simulated and the observed points are; the region colouring, the `ksreg` bins and the
  `ks_stage_regions.csv` counts all refer to it. Kennicutt 98 (total gas, slope 1.4) is drawn thin
  (`RELATIONS_FAINT`) for orientation and kept as a numerical offset in `ks_stage_summary` with RK19; Figure 3(d)
  compares it with the total neutral gas, the quantity it was calibrated on. Tacconi+18 main-sequence t$_{\rm dep}$ at z = 0.37.
* **Fair comparison** (Figure 3, `ks-54-fig-fair`): with one radius for both Σ the offset from K98 is
  Δ = (6 − A$_{\rm K98}$) − log t$_{\rm dep}$ − 0.4 log Σ$_{\rm H_2}$, so the KS position is the H$_2$ depletion time; SIMBA's
  SF law (SFR = ε$_{\rm ff}$ ρ$_{\rm H_2}$/t$_{\rm ff}$, ε$_{\rm ff}$ = 0.02, on the same subgrid f$_{\rm H_2}$) fixes it at
  t$_{\rm ff}$/ε$_{\rm ff}$ ≈ 0.7–6 Gyr, hence few galaxies below a 1.4-slope total-gas relation whatever the aperture. The
  fair references: Bigiel+08 (H$_2$, ±0.20 dex) for Σ$_{\rm H_2}$, and K98/RK19 for the total neutral gas
  `logSigmaGas` = (m$_{\rm HI}$ + m$_{\rm H_2}$)(<R) × 1.36 / πR² (caesar HI/H2 split; ionised gas excluded; observed points
  = lower limits there). Observed t$_{\rm dep}$ carries α$_{\rm CO}$ = 4.36 and R$_{31}$ = 0.5 — a lower α$_{\rm CO}$ halves it.
* **Products** (`output/cis25/ks_tracks/`): `ks_track_epochs.fits` (Part 1), `ks_track_measurements.fits`
  (Part 3), `ks_tracks.fits` + `ks_track_nodes.csv` (Part 4; the CSV uses the observed table's column
  names so `pilot_specphot/scripts/plot_ks.py` can overlay it), `ks_stage_summary.fits/.csv`, `ks_fair_comparison.csv`,
  `ks_track_histories.fits` + `ks_region_*.csv` (Part 5).
  Figures in `output/cis25/plots/ks_tracks/`.